## Generating statistics to make the script more realistic

In [76]:
import os
import json
import numpy as np
from collections import defaultdict
from typing import Dict, List, Any

In [77]:
def as_list_on_duplicate_keys(ordered_pairs):
    d = {}
    for k, v in ordered_pairs:
        if k in d:
            if isinstance(d[k], list): d[k].append(v)
            else: d[k] = [d[k], v]
        else: d[k] = v
    return d

In [78]:
def calculate_statistics_from_captures(directory_path: str) -> dict:
    """
    Analyzes QUIC capture JSON files to build comprehensive statistical profiles.
    
    Extracts statistics for:
    - Packet sizes (handshake, data, ACKs, control frames, HTTP/3 streams)
    - Delta times (handshake, data transfer, ACKs)
    - ACK frequency patterns
    - Path validation (PATH_CHALLENGE/RESPONSE)
    - Connection migration timing
    
    Args:
        directory_path: Path to directory containing Wireshark JSON exports
        
    Returns:
        Dictionary with statistical profiles (mean, std, sample count)
    """
    raw_stats = defaultdict(lambda: defaultdict(list))
    
    print(f"Analyzing JSON files in: {directory_path}\n")
    
    for filename in os.listdir(directory_path):
        if not filename.endswith('.json'):
            continue
            
        json_file_path = os.path.join(directory_path, filename)
        print(f"  -> Processing: {filename}")
        
        # Load JSON with encoding fallback
        try:
            with open(json_file_path, 'r', encoding='utf-8') as f:
                packets = json.load(f, object_pairs_hook=as_list_on_duplicate_keys)
        except (json.JSONDecodeError, UnicodeDecodeError):
            try:
                with open(json_file_path, 'r', encoding='utf-16') as f:
                    packets = json.load(f, object_pairs_hook=as_list_on_duplicate_keys)
            except Exception as e:
                print(f"     ... Skipping, could not decode JSON: {e}")
                continue
        
        if not packets:
            continue
        
        # Extract initial connection parameters
        try:
            first_packet_layers = packets[0]['_source']['layers']
            if not ('ip' in first_packet_layers and 'frame' in first_packet_layers):
                continue
                
            initial_ip_client = first_packet_layers['ip']['ip.src']
            initial_ip_server = first_packet_layers['ip']['ip.dst']
            initial_port_client = int(first_packet_layers['udp']['udp.srcport'])
            initial_port_server = int(first_packet_layers['udp']['udp.dstport'])
            last_packet_time = float(first_packet_layers['frame']['frame.time_epoch'])
        except (KeyError, IndexError):
            print(f"     ... Skipping, missing required fields")
            continue
        
        # State tracking
        in_handshake = True
        migrated = False
        handshake_done_seen = False
        client_data_since_server_ack = 0
        server_data_since_client_ack = 0
        
        # Process each packet
        for pkt_index, pkt_data in enumerate(packets):
            layers = pkt_data.get('_source', {}).get('layers', {})
            if not ('ip' in layers and 'frame' in layers):
                continue
            
            # Extract packet metadata
            src_ip = layers['ip']['ip.src']
            dst_ip = layers['ip']['ip.dst']
            src_port = int(layers['udp']['udp.srcport'])
            dst_port = int(layers['udp']['udp.dstport'])
            
            # Detect migration
            if not migrated and \
               (dst_ip == initial_ip_server and dst_port == initial_port_server) and \
               (src_ip != initial_ip_client or src_port != initial_port_client):
                migrated = True
                raw_stats['behavior_counts']['packets_before_migration'].append(pkt_index)
            
            # Calculate timing
            current_time = float(layers['frame']['frame.time_epoch'])
            delta_time = current_time - last_packet_time
            last_packet_time = current_time
            
            # Direction
            is_client_pkt = (src_ip == initial_ip_client and src_port == initial_port_client)
            direction = 'client' if is_client_pkt else 'server'
            
            # Packet length
            pkt_len = int(layers['frame']['frame.len'])
            
            # Extract QUIC layer
            quic_packet_list = layers.get('quic', [])
            if not isinstance(quic_packet_list, list):
                quic_packet_list = [quic_packet_list]
            
            # Process each QUIC packet (can have multiple per UDP packet)
            for quic_packet in quic_packet_list:
                # Get packet type
                packet_type = quic_packet.get('quic.long.packet_type')
                is_long_header = quic_packet.get('quic.header_form') == '1'
                
                # Extract frames
                quic_frames = quic_packet.get('quic.frame', [])
                if not isinstance(quic_frames, list):
                    quic_frames = [quic_frames]
                
                # Analyze frame types
                frame_types = {f.get('quic.frame_type', '0') for f in quic_frames}
                
                # Frame type checks
                has_crypto = '0x0000000000000006' in frame_types
                has_ack = '0x0000000000000002' in frame_types or '0x0000000000000003' in frame_types
                has_stream = any('0x0000000000000008' <= ft <= '0x000000000000000f' for ft in frame_types)
                has_padding = '0x0000000000000000' in frame_types
                has_ping = '0x0000000000000001' in frame_types
                has_connection_close = '0x000000000000001c' in frame_types or '0x000000000000001d' in frame_types
                has_path_challenge = '0x000000000000001a' in frame_types
                has_path_response = '0x000000000000001b' in frame_types
                has_new_connection_id = '0x0000000000000018' in frame_types
                has_handshake_done = '0x000000000000001e' in frame_types
                
                # Detect handshake completion
                if has_handshake_done:
                    handshake_done_seen = True
                    in_handshake = False
                
                # ============================================================
                # HANDSHAKE PHASE STATISTICS
                # ============================================================
                if in_handshake or (is_long_header and has_crypto):
                    # Delta times
                    delta_key = 'handshake_c2s' if is_client_pkt else 'handshake_s2c'
                    raw_stats['delta_times'][delta_key].append(delta_time)
                    
                    # Packet sizes
                    if packet_type == '0' and is_client_pkt:
                        # Initial packet from client
                        raw_stats['packet_sizes']['handshake_initial_client'].append(pkt_len)
                    elif packet_type == '0' and not is_client_pkt:
                        # Initial packet from server
                        raw_stats['packet_sizes']['handshake_initial_server'].append(pkt_len)
                    elif packet_type == '2':
                        # Handshake packet
                        raw_stats['packet_sizes'][f'handshake_handshake_{direction}'].append(pkt_len)
                    elif packet_type == '3':
                        # Retry packet
                        raw_stats['packet_sizes']['handshake_retry'].append(pkt_len)
                    else:
                        # Other handshake packets
                        raw_stats['packet_sizes'][f'handshake_other_{direction}'].append(pkt_len)
                
                # ============================================================
                # 1-RTT PHASE STATISTICS
                # ============================================================
                elif not in_handshake or handshake_done_seen:
                    # PATH_CHALLENGE/RESPONSE (connection migration)
                    if has_path_challenge:
                        raw_stats['packet_sizes']['path_challenge'].append(pkt_len)
                        raw_stats['delta_times']['path_challenge'].append(delta_time)
                    
                    if has_path_response:
                        raw_stats['packet_sizes']['path_response'].append(pkt_len)
                        raw_stats['delta_times']['path_response'].append(delta_time)
                    
                    # ACK-only packets
                    if has_ack and not has_stream and not has_connection_close and not has_path_challenge and not has_path_response:
                        raw_stats['packet_sizes'][f'ack_{direction}'].append(pkt_len)
                        raw_stats['delta_times']['ack_response'].append(delta_time)
                        
                        # Track ACK frequency
                        if is_client_pkt:
                            if server_data_since_client_ack > 0:
                                raw_stats['ack_frequency']['client_sends_ack_after'].append(server_data_since_client_ack)
                            server_data_since_client_ack = 0
                        else:
                            if client_data_since_server_ack > 0:
                                raw_stats['ack_frequency']['server_sends_ack_after'].append(client_data_since_server_ack)
                            client_data_since_server_ack = 0
                    
                    # PING-only packets
                    elif has_ping and not has_stream and not has_connection_close:
                        raw_stats['packet_sizes'][f'ping_{direction}'].append(pkt_len)
                        raw_stats['delta_times'][f'ping_{direction}'].append(delta_time)
                    
                    # CONNECTION_CLOSE packets
                    elif has_connection_close:
                        raw_stats['packet_sizes'][f'close_{direction}'].append(pkt_len)
                        raw_stats['delta_times']['close'].append(delta_time)
                    
                    # STREAM packets (application data)
                    elif has_stream:
                        delta_key = 'client_request' if is_client_pkt else 'server_response'
                        raw_stats['delta_times'][delta_key].append(delta_time)
                        
                        # Track for ACK frequency
                        if is_client_pkt:
                            client_data_since_server_ack += 1
                        else:
                            server_data_since_client_ack += 1
                        
                        # Categorize by stream type (based on stream ID)
                        stream_type_found = False
                        for frame in quic_frames:
                            frame_type = frame.get('quic.frame_type', '0')
                            if '0x0000000000000008' <= frame_type <= '0x000000000000000f':
                                stream_id = frame.get('quic.stream.stream_id')
                                
                                if stream_id is not None:
                                    try:
                                        stream_id = int(stream_id)
                                        
                                        # Determine stream type from stream ID
                                        # Stream ID modulo 4 determines initiator and directionality
                                        if stream_id % 4 == 0:
                                            stream_type_key = 'pkt_size_bidi_client'
                                        elif stream_id % 4 == 1:
                                            stream_type_key = 'pkt_size_bidi_server'
                                        elif stream_id % 4 == 2:
                                            stream_type_key = 'pkt_size_uni_client'
                                        elif stream_id % 4 == 3:
                                            stream_type_key = 'pkt_size_uni_server'
                                        
                                        raw_stats['packet_sizes'][stream_type_key].append(pkt_len)
                                        stream_type_found = True
                                        break  # Use first stream in packet
                                    except (ValueError, TypeError):
                                        continue
                        
                        # Fallback if no stream ID found
                        if not stream_type_found:
                            raw_stats['packet_sizes'][f'stream_data_{direction}'].append(pkt_len)
                    
                    # Mixed packets (have ACK + data)
                    elif has_ack and has_stream:
                        # Already counted in stream category above
                        pass
    
    # ============================================================
    # COMPUTE FINAL STATISTICS
    # ============================================================
    final_stats = defaultdict(dict)
    
    for category, keys in raw_stats.items():
        for key, data_list in keys.items():
            # Filter out invalid values
            if 'ack_frequency' in category:
                data_list = [x for x in data_list if x > 0]
            
            if 'delta_times' in category:
                data_list = [x for x in data_list if x >= 0]
            
            if 'packet_sizes' in category:
                data_list = [x for x in data_list if x > 0]
            
            # Calculate statistics
            if data_list and len(data_list) > 0:
                final_stats[category][key] = {
                    'mean': float(np.mean(data_list)),
                    'std': float(np.std(data_list)),
                    'min': float(np.min(data_list)),
                    'max': float(np.max(data_list)),
                    'median': float(np.median(data_list)),
                    'samples': len(data_list)
                }
            else:
                final_stats[category][key] = {
                    'mean': 0.0,
                    'std': 0.0,
                    'min': 0.0,
                    'max': 0.0,
                    'median': 0.0,
                    'samples': 0
                }
    
    print(f"\n=== Statistics Summary ===")
    print(f"Packet size categories: {len(final_stats.get('packet_sizes', {}))}")
    print(f"Delta time categories: {len(final_stats.get('delta_times', {}))}")
    print(f"ACK frequency metrics: {len(final_stats.get('ack_frequency', {}))}")
    print(f"Behavior counts: {len(final_stats.get('behavior_counts', {}))}")
    
    return dict(final_stats)


def print_statistics_summary(stats: Dict[str, Any]) -> None:
    """Pretty print statistics summary."""
    print("\n" + "="*80)
    print("STATISTICS SUMMARY")
    print("="*80)
    
    for category, metrics in stats.items():
        print(f"\n{category.upper()}:")
        print("-" * 80)
        
        for key, values in sorted(metrics.items()):
            if values['samples'] > 0:
                print(f"  {key:40s} | μ={values['mean']:8.4f}  σ={values['std']:8.4f}  "
                      f"n={values['samples']:4d}  [{values['min']:.2f}, {values['max']:.2f}]")
            else:
                print(f"  {key:40s} | No samples")


In [43]:
def calculate_statistics_from_captures(directory_path: str) -> dict:
    """
    Analyzes all captures to build the final, hyper-realistic statistical profile.
    Includes packet count before migration and deep HTTP/3 analysis.
    """
    raw_stats = defaultdict(lambda: defaultdict(list))

    print(f"Analyzing JSON files in: {directory_path}\n")
    for filename in os.listdir(directory_path):
        if not filename.endswith('.json'): continue
        json_file_path = os.path.join(directory_path, filename)
        print(f"  -> Processing: {filename}")
        
        try: # Your robust file loading logic
            with open(json_file_path, 'r', encoding='utf-8') as f:
                packets = json.load(f, object_pairs_hook=as_list_on_duplicate_keys)
        except (json.JSONDecodeError, UnicodeDecodeError):
            try:
                with open(json_file_path, 'r', encoding='utf-16') as f:
                    packets = json.load(f, object_pairs_hook=as_list_on_duplicate_keys)
            except Exception as e:
                print(f"     ... Skipping, could not decode JSON: {e}"); continue
        
        if not packets: continue

        try:
            first_packet_layers = packets[0]['_source']['layers']
            if not ('ip' in first_packet_layers and 'frame' in first_packet_layers): continue
            initial_ip_client = first_packet_layers['ip']['ip.src']
            initial_ip_server = first_packet_layers['ip']['ip.dst']
            initial_port_client = int(first_packet_layers['udp']['udp.srcport'])
            initial_port_server = int(first_packet_layers['udp']['udp.dstport'])
            last_packet_time = float(first_packet_layers['frame']['frame.time_epoch'])
        except (KeyError, IndexError): continue

        in_handshake = True 
        migrated = False
        client_data_since_server_ack = 0
        server_data_since_client_ack = 0

        for i, pkt_data in enumerate(packets):
            layers = pkt_data.get('_source', {}).get('layers', {})
            if not ('ip' in layers and 'frame' in layers): continue

            # --- NEW: Track migration by packet count ---
            src_ip = layers['ip']['ip.src']
            dst_ip = layers['ip']['ip.dst']

            src_port = int(layers['udp']['udp.srcport'])
            dst_port = int(layers['udp']['udp.dstport'])

            if not migrated and \
                (dst_ip == initial_ip_server and dst_port == initial_port_server) and \
                (src_ip != initial_ip_client or src_port != initial_port_client):
                migrated = True
                raw_stats['behavior_counts']['packets_before_migration'].append(i)
            
            current_time = float(layers['frame']['frame.time_epoch'])
            delta_time_msec = (current_time - last_packet_time) 
            last_packet_time = current_time
            is_client_pkt = layers['ip']['ip.src'] == initial_ip_client
            direction = 'client' if is_client_pkt else 'server'
            pkt_len = int(layers['frame']['frame.len'])
            quic_packet_list = layers.get('quic', [])
            if not isinstance(quic_packet_list, list): quic_packet_list = [quic_packet_list]

            for quic_packet in quic_packet_list:
                quic_frames = quic_packet.get('quic.frame', [])
                if not isinstance(quic_frames, list): quic_frames = [quic_frames]
                
                frame_types = {f.get('quic.frame_type') for f in quic_frames}
                has_http3_settings = False; has_http3_headers = False; has_http3_data = False
                for frame in quic_frames:
                    if '0x0000000000000008' <= frame.get('quic.frame_type', '0') <= '0x000000000000000f':
                        if frame.get('http3.settings'): has_http3_settings = True
                        elif frame.get('http3.headers'): has_http3_headers = True
                        elif frame.get('http3.data'): has_http3_data = True
                
                has_stream_frame = any('0x0000000000000008' <= ft <= '0x000000000000000f' for ft in frame_types)
                has_ack = '0x0000000000000002' in frame_types or '0x0000000000000003' in frame_types
                is_close = '0x000000000000001c' in frame_types or '0x000000000000001d' in frame_types
                is_ping_only = '0x0000000000000001' in frame_types and len(frame_types) == 1
                is_ack_only = has_ack and not has_stream_frame and not is_close

                if in_handshake:
                    delta_key = 'handshake_c2s' if is_client_pkt else 'handshake_s2c'
                    raw_stats['delta_times'][delta_key].append(delta_time_msec)
                    if quic_packet.get('quic.long.packet_type') == '0' and is_client_pkt:
                        raw_stats['packet_sizes']['handshake_initial_client'].append(pkt_len)
                    else:
                        raw_stats['packet_sizes'][f'handshake_other_{direction}'].append(pkt_len)
                    if '0x000000000000001e' in frame_types: in_handshake = False
                else: # 1-RTT phase
                    if is_ack_only:
                        raw_stats['packet_sizes'][f'ack_{direction}'].append(pkt_len)
                        raw_stats['delta_times']['ack_response'].append(delta_time_msec)
                        if is_client_pkt:
                            if server_data_since_client_ack > 0: raw_stats['ack_frequency']['client_sends_ack_after'].append(server_data_since_client_ack)
                            server_data_since_client_ack = 0
                        else:
                            if client_data_since_server_ack > 0: raw_stats['ack_frequency']['server_sends_ack_after'].append(client_data_since_server_ack)
                            client_data_since_server_ack = 0
                    elif is_ping_only:
                        raw_stats['packet_sizes'][f'ping_{direction}'].append(pkt_len)
                    elif is_close:
                        raw_stats['packet_sizes'][f'close_{direction}'].append(pkt_len)
                    elif has_stream_frame:
                        delta_key = 'client_request' if is_client_pkt else 'server_response'
                        raw_stats['delta_times'][delta_key].append(delta_time_msec)
                        if is_client_pkt: client_data_since_server_ack += 1
                        else: server_data_since_client_ack += 1

                        # Identify the dominant stream type in the packet
                        # For simplicity, we'll categorize the packet by the first stream type found
                        dominant_stream_type_key = None
                        for frame in quic_frames:
                            if '0x0000000000000008' <= frame.get('quic.frame_type', '0') <= '0x000000000000000f':
                                stream_id = int(frame.get('quic.stream.stream_id', -1))
                                if stream_id == -1: continue
                                
                                if stream_id % 4 == 0: dominant_stream_type_key = 'pkt_size_bidi_client'
                                elif stream_id % 4 == 1: dominant_stream_type_key = 'pkt_size_bidi_server'
                                elif stream_id % 4 == 2: dominant_stream_type_key = 'pkt_size_uni_client'
                                elif stream_id % 4 == 3: dominant_stream_type_key = 'pkt_size_uni_server'
                                
                                # Once we've identified the type, we categorize the whole packet and stop.
                                break 
                        
                        if dominant_stream_type_key:
                            raw_stats['packet_sizes'][dominant_stream_type_key].append(pkt_len)

    final_stats = defaultdict(dict)
    for category, keys in raw_stats.items():
        for key, data in keys.items():
            if 'ack_frequency' in category: data = [x for x in data if x > 0]
            if data:
                final_stats[category][key] = {'mean': float(np.mean(data)), 'std': float(np.std(data)), 'samples': len(data)}
            else:
                final_stats[category][key] = {'mean': 0, 'std': 0, 'samples': 0}
    return final_stats

In [79]:
json_captures_directory = "C:/Users/vassa/Desktop/UZH/Masters Project/synthetic_network_data_gen/captures_json/quiche"

if not os.path.isdir(json_captures_directory):
    print(f"Error: Directory not found at '{json_captures_directory}'")
    print("Please update the 'json_captures_directory' variable with the correct path.")
else:
    full_stats_profile = calculate_statistics_from_captures(json_captures_directory)
    
    print("\n\n--- Calculated Statistical Profile ---")
    print(json.dumps(full_stats_profile, indent=4))

Analyzing JSON files in: C:/Users/vassa/Desktop/UZH/Masters Project/synthetic_network_data_gen/captures_json/quiche

  -> Processing: quiche_capture_1.json
  -> Processing: quiche_capture_10.json
  -> Processing: quiche_capture_11.json
  -> Processing: quiche_capture_12.json
  -> Processing: quiche_capture_13.json
  -> Processing: quiche_capture_14.json
  -> Processing: quiche_capture_15.json
  -> Processing: quiche_capture_16.json
  -> Processing: quiche_capture_17.json
  -> Processing: quiche_capture_18.json
  -> Processing: quiche_capture_19.json
  -> Processing: quiche_capture_2.json
  -> Processing: quiche_capture_20.json
  -> Processing: quiche_capture_21.json
  -> Processing: quiche_capture_22.json
  -> Processing: quiche_capture_23.json
  -> Processing: quiche_capture_24.json
  -> Processing: quiche_capture_25.json
  -> Processing: quiche_capture_26.json
  -> Processing: quiche_capture_27.json
  -> Processing: quiche_capture_28.json
  -> Processing: quiche_capture_29.json
  -> 

In [80]:
print_statistics_summary(full_stats_profile)


STATISTICS SUMMARY

DELTA_TIMES:
--------------------------------------------------------------------------------
  ack_response                             | μ=  0.0001  σ=  0.0002  n= 343  [0.00, 0.00]
  close                                    | μ=  0.0006  σ=  0.0004  n=  90  [0.00, 0.00]
  handshake_c2s                            | μ=  0.0012  σ=  0.0013  n= 720  [0.00, 0.01]
  handshake_s2c                            | μ=  0.0020  σ=  0.0044  n= 450  [0.00, 0.06]
  path_challenge                           | μ=  0.0004  σ=  0.0003  n= 180  [0.00, 0.00]
  path_response                            | μ=  0.0004  σ=  0.0003  n= 180  [0.00, 0.00]
  server_response                          | μ=  0.0003  σ=  0.0006  n= 900  [0.00, 0.02]

PACKET_SIZES:
--------------------------------------------------------------------------------
  ack_client                               | μ= 75.0000  σ=  0.0000  n= 168  [75.00, 75.00]
  ack_server                               | μ= 75.0457  σ=  0.2089

In [74]:
full_stats_profile

{'delta_times': {'handshake_c2s': {'mean': 0.00122932857937283,
   'std': 0.0012916642172444382,
   'min': 0.0,
   'max': 0.005407094955444336,
   'median': 0.0007650852203369141,
   'samples': 720},
  'handshake_s2c': {'mean': 0.002035429212782118,
   'std': 0.004433845833301648,
   'min': 2.8848648071289062e-05,
   'max': 0.05875706672668457,
   'median': 0.0003104209899902344,
   'samples': 450},
  'server_response': {'mean': 0.0002897450659010145,
   'std': 0.0006496823374970188,
   'min': 9.5367431640625e-07,
   'max': 0.015305042266845703,
   'median': 0.00010192394256591797,
   'samples': 900},
  'ack_response': {'mean': 0.0001490484521270841,
   'std': 0.00015428837286079192,
   'min': 0.0,
   'max': 0.001444101333618164,
   'median': 9.298324584960938e-05,
   'samples': 343},
  'path_challenge': {'mean': 0.0003615379333496094,
   'std': 0.00031815651292679305,
   'min': 5.0067901611328125e-06,
   'max': 0.0015590190887451172,
   'median': 0.00025594234466552734,
   'samples': 

#### Generating low level features with the statistics

In [48]:
import numpy as np
import csv
import pprint
import pandas as pd

SIMULATED_MTU = 1350

def generate_statistically_realistic_features(blueprint: dict, stats: dict) -> list:
    """
    Generates a statistically realistic sequence of low-level packet features,
    using a profile of means and standard deviations from real data and ensuring
    byte totals from the blueprint are met exactly.
    """
    output_packet_features = []
    current_time_msec = 0.0
    current_packet_count = 0
    client_bytes_sent = 0
    server_bytes_sent = 0
    has_migrated = False
    time_of_last_packet = 0.0
    client_ack_pending = False
    server_ack_pending = False

    client_bidi_streams_to_send = blueprint.get('client_bidi_streams_count', 0)
    client_uni_streams_to_send = blueprint.get('client_uni_streams_count', 0)
    server_bidi_streams_to_send = 0 # Assuming server bidi is rare for this model
    server_uni_streams_to_send = blueprint.get('server_uni_streams_count', 0)

    # Helper to safely draw a random number from the profile, with a fallback
    def get_stat(category, key, fallback_mean=0, fallback_std=0):
        if category in stats and key in stats[category] and stats[category][key]['samples'] > 0:
            mean = stats[category][key]['mean']
            std = stats[category][key]['std']
            return max(1, np.random.normal(mean, std))
        return max(1, np.random.normal(fallback_mean, fallback_std))

    def create_low_level_feature(delta_time, length, direction, quic_type, frame_flags=None):
        nonlocal current_packet_count
        current_packet_count += 1
        if frame_flags is None: frame_flags = {}
        feature_vector = {'delta_time': round(delta_time, 4), 'packet_length': int(length), 'packet_direction': direction, 'header_form': 1 if quic_type in ['is_initial', 'is_handshake', 'is_0rtt', 'is_retry'] else 0, 'quic_packet_type_is_initial': 0, 'quic_packet_type_is_handshake': 0, 'quic_packet_type_is_0rtt': 0, 'quic_packet_type_is_1rtt': 0, 'quic_packet_type_is_retry': 0, 'quic_packet_type_is_vn': 0, 'has_path_challenge': 0, 'has_path_response': 0, 'has_new_connection_id': 0, 'has_retire_cid': 0, 'has_padding': 0, 'has_ack': 0, 'has_connection_close': 0, 'http3_stream_count': 0, 'http3_fin_count': 0, 'has_ping': 0}
        feature_vector[f'quic_packet_type_{quic_type}'] = 1
        feature_vector.update(frame_flags)
        return feature_vector

    # --- HANDSHAKE PHASE using stats ---
    # Simplified for clarity, but demonstrates stat usage
    handshake_duration = blueprint.get('handshake_duration_msec', 100.0) or 100.0
    
    handshake_duration = blueprint.get('handshake_duration_msec', 100.0) or 100.0
    if blueprint.get('retry_occurred', 0) == 1:
        # Client Initial
        output_packet_features.append(create_low_level_feature(get_stat('delta_times', 'handshake_c2s', 15, 5), get_stat('packet_sizes', 'handshake_initial_client', 1250, 50), 0, 'is_initial'))
        # Server Retry
        output_packet_features.append(create_low_level_feature(get_stat('delta_times', 'handshake_s2c', 2, 1), get_stat('packet_sizes', 'handshake_other_server', 100, 20), 1, 'is_retry'))
        # New Client Initial
        output_packet_features.append(create_low_level_feature(get_stat('delta_times', 'handshake_c2s', 15, 5), get_stat('packet_sizes', 'handshake_initial_client', 1250, 50), 0, 'is_initial'))
        # Server Handshake
        output_packet_features.append(create_low_level_feature(get_stat('delta_times', 'handshake_s2c', 2, 1), get_stat('packet_sizes', 'handshake_other_server', 1000, 200), 1, 'is_handshake'))
        # Client Handshake
        output_packet_features.append(create_low_level_feature(get_stat('delta_times', 'handshake_c2s', 15, 5), get_stat('packet_sizes', 'handshake_other_client', 300, 100), 0, 'is_handshake', {'has_ack': 1}))
        # Server Handshake Done
        output_packet_features.append(create_low_level_feature(get_stat('delta_times', 'handshake_s2c', 2, 1), get_stat('packet_sizes', 'ack_server', 96, 10), 1, 'is_1rtt', {'has_ack': 1}))
    else: # Simplified non-retry
        # ... (similar logic using get_stat for a 4-packet handshake) ...
        pass
    # For simplicity, we'll just set the time after the handshake based on the blueprint
    current_time_msec = handshake_duration
    time_of_last_packet = handshake_duration

    packets_to_migrate = blueprint.get('packets_before_migration', float('inf'))
    
    ack_countdown = get_stat('ack_frequency', 'server_sends_ack_after', 2, 1)

    # --- MAIN LOOP with stats and guaranteed byte totals ---
    while current_time_msec < blueprint['connection_duration_msec']:
        made_progress_this_cycle = False
        
        # PRIORITY 1: MIGRATION EVENT
        if not has_migrated and current_time_msec >= blueprint.get('time_to_migration_msec', float('inf')):
            has_migrated = True; made_progress_this_cycle = True
            validation_rtt = blueprint.get('migration_validation_duration_msec', 50.0) or 50.0
            output_packet_features.append(create_low_level_feature(get_stat('delta_times', 'client_request', 5, 2), SIMULATED_MTU, 0, 'is_1rtt', {'has_path_challenge': 1, 'has_padding': 1}))
            output_packet_features.append(create_low_level_feature(get_stat('delta_times', 'server_response', 5, 2), SIMULATED_MTU, 1, 'is_1rtt', {'has_path_challenge': 1, 'has_padding': 1}))
            output_packet_features.append(create_low_level_feature(validation_rtt, SIMULATED_MTU, 0, 'is_1rtt', {'has_path_response': 1, 'has_padding': 1}))
            output_packet_features.append(create_low_level_feature(get_stat('delta_times', 'ack_response', 2, 1), SIMULATED_MTU, 1, 'is_1rtt', {'has_path_response': 1, 'has_padding': 1}))
            current_time_msec += validation_rtt + 10; time_of_last_packet = current_time_msec
            continue

        # PRIORITY 2: CLIENT DATA TRANSFER
        elif client_bidi_streams_to_send > 0 or client_uni_streams_to_send > 0:
            made_progress_this_cycle = True
            
            # Prioritize sending bidirectional streams (like GET requests)
            if client_bidi_streams_to_send > 0:
                size = get_stat('packet_sizes', 'pkt_size_bidi_client', 150, 50)
                client_bidi_streams_to_send -= 1
            else: # Send a unidirectional stream
                size = get_stat('packet_sizes', 'pkt_size_uni_client', 80, 15)
                client_uni_streams_to_send -= 1
            
            frame_flags = {'http3_stream_count': 1}
            if server_ack_pending: frame_flags['has_ack'] = 1; server_ack_pending = False
            
            delta = get_stat('delta_times', 'client_request', 20, 8)
            output_packet_features.append(create_low_level_feature(delta, size, 0, 'is_1rtt', frame_flags))
            current_time_msec += delta; time_of_last_packet = current_time_msec; client_ack_pending = True
        
        # PRIORITY 3: SERVER sends a stream packet
        elif server_bidi_streams_to_send > 0 or server_uni_streams_to_send > 0 or server_bytes_sent < blueprint['total_server_app_bytes']:
            made_progress_this_cycle = True
            
            # Prioritize unidirectional streams (like HTTP/3 SETTINGS)
            if server_uni_streams_to_send > 0:
                size = get_stat('packet_sizes', 'pkt_size_uni_server', 100, 20)
                server_uni_streams_to_send -= 1
            else: # Then send bidirectional data (like the response body)
                bytes_remaining = blueprint['total_server_app_bytes'] - server_bytes_sent
                # Use the bidi_server key which typically carries the response data
                size = get_stat('packet_sizes', 'pkt_size_bidi_server', 800, 300)
                actual_chunk_size = min(size, bytes_remaining + 40) # Add overhead
                server_bytes_sent += max(0, actual_chunk_size - 40)
                size = actual_chunk_size

            frame_flags = {'http3_stream_count': 1}
            if client_ack_pending: frame_flags['has_ack'] = 1; client_ack_pending = False

            delta = get_stat('delta_times', 'server_response', 15, 5)
            output_packet_features.append(create_low_level_feature(delta, size, 1, 'is_1rtt', frame_flags))
            current_time_msec += delta; time_of_last_packet = current_time_msec; server_ack_pending = True


        # PRIORITY 4: PROTOCOL CHATTINESS
        if not made_progress_this_cycle:
            if client_ack_pending:
                delta = get_stat('delta_times', 'ack_response', 2, 1); size = get_stat('packet_sizes', 'ack_server', 65, 5)
                output_packet_features.append(create_low_level_feature(delta, size, 1, 'is_1rtt', {'has_ack': 1})); current_time_msec += delta; time_of_last_packet = current_time_msec; client_ack_pending = False
            elif server_ack_pending:
                delta = get_stat('delta_times', 'ack_response', 2, 1); size = get_stat('packet_sizes', 'ack_client', 65, 5)
                output_packet_features.append(create_low_level_feature(delta, size, 0, 'is_1rtt', {'has_ack': 1})); current_time_msec += delta; time_of_last_packet = current_time_msec; server_ack_pending = False
            else: # Idle
                current_time_msec += 50.0


        if ack_countdown <= 0:
            delta = get_stat('delta_times', 'ack_response', 2, 1)
            size = get_stat('packet_sizes', 'ack_server', 65, 5) # Can be client or server
            output_packet_features.append(create_low_level_feature(delta, size, np.random.choice([0,1]), 'is_1rtt', {'has_ack': 1}))
            current_time_msec += delta; time_of_last_packet = current_time_msec
            ack_countdown = get_stat('ack_frequency', 'server_sends_ack_after', 2, 1)

    # --- TEARDOWN PHASE ---
    close_type = blueprint.get('connection_close_type', 'IDLE_TIMEOUT')
    if close_type != 'IDLE_TIMEOUT':
        close_direction = 1 if close_type == 'SERVER_CLOSE' else 0
        delta = get_stat('delta_times', 'close', 50, 10)
        size = get_stat('packet_sizes', f'close_{"server" if close_direction == 1 else "client"}', 62, 4)
        output_packet_features.append(create_low_level_feature(delta, size, close_direction, 'is_1rtt', {'has_connection_close': 1}))
    
    return output_packet_features


In [58]:
real_world_blueprint = {
        "connection_duration_msec": 107.25, "retry_occurred": 1, "server_issued_cid_count": 1, "migration_type": "IP_AND_PORT", "connection_close_type": "CLIENT_CLOSE",
        "handshake_duration_msec": 77.66, "total_client_app_bytes": 119, "total_server_app_bytes": 355, "avg_request_size": 23.8, "avg_response_size": 71.0,
        "client_bidi_streams_count": 1, "client_uni_streams_count": 4, "server_uni_streams_count": 4, "time_to_migration_msec": 86.83, "app_data_bytes_before_migration": 47, "migration_validation_duration_msec": 1.17
    }

print("--- Generating features with STATISTICALLY REALISTIC script (Byte Totals Guaranteed) ---")
low_level_packets = generate_statistically_realistic_features(real_world_blueprint, full_stats_profile)
df = pd.DataFrame(low_level_packets)
def create_summary(row):
    info = ['C->S' if row['packet_direction'] == 0 else 'S->C']
    for col, name in [('quic_packet_type_is_initial', 'Initial'), ('quic_packet_type_is_handshake', 'Handshake'),
                        ('quic_packet_type_is_1rtt', '1-RTT'), ('quic_packet_type_is_retry', 'Retry')]:
        if row[col] == 1: info.append(name); break
    for col, name in [('has_new_connection_id', 'NEW_CID'), ('has_path_challenge', 'PATH_CHALLENGE'),
                        ('has_path_response', 'PATH_RESPONSE'), ('has_ack', 'ACK'),
                        ('has_connection_close', 'CLOSE'), ('has_ping', 'PING')]:
        if row[col] > 0: info.append(name)
    if row['has_padding'] == 1: info.append('PADDED')
    if row['http3_stream_count'] > 0: info.append(f"STREAM({row['http3_stream_count']})")
    return ', '.join(info)

#df['Summary'] = df.apply(create_summary, axis=1)
# print(f"\n--- Generated {len(df)} packets. Displaying key columns: ---")
# display_columns = ['delta_time', 'packet_length', 'Summary']
# with pd.option_context('display.max_rows', None, 'display.width', 1000):
#         print(df[display_columns])
display(df)

--- Generating features with STATISTICALLY REALISTIC script (Byte Totals Guaranteed) ---


,delta_time,packet_length,packet_direction,header_form,quic_packet_type_is_initial,quic_packet_type_is_handshake,quic_packet_type_is_0rtt,quic_packet_type_is_1rtt,quic_packet_type_is_retry,quic_packet_type_is_vn,has_path_challenge,has_path_response,has_new_connection_id,has_retire_cid,has_padding,has_ack,has_connection_close,http3_stream_count,http3_fin_count,has_ping
0,1.0000,1204,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1.0000,614,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
2,1.0000,1300,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,1.0000,864,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,1.0000,1148,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
5,1.0000,78,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0
6,21.4768,207,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0
7,4.8134,1350,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0
8,1.0000,1350,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0
9,1.1700,1350,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0


In [50]:
df

,delta_time,packet_length,packet_direction,header_form,quic_packet_type_is_initial,quic_packet_type_is_handshake,quic_packet_type_is_0rtt,quic_packet_type_is_1rtt,quic_packet_type_is_retry,quic_packet_type_is_vn,has_path_challenge,has_path_response,has_new_connection_id,has_retire_cid,has_padding,has_ack,has_connection_close,http3_stream_count,http3_fin_count,has_ping,Summary
0,1.0000,1351,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"C->S, Initial"
1,1.0000,467,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,"S->C, Retry"
2,1.0000,1251,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"C->S, Initial"
3,1.0000,718,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"S->C, Handshake"
4,1.0000,1238,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,"C->S, Handshake, ACK"
5,1.0000,77,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,"S->C, 1-RTT, ACK"
6,24.2368,268,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,"C->S, 1-RTT, STREAM(1.0)"
7,5.5200,1350,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,"C->S, 1-RTT, PATH_CHALLENGE, PADDED"
8,1.0000,1350,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,"S->C, 1-RTT, PATH_CHALLENGE, PADDED"
9,1.1700,1350,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,"C->S, 1-RTT, PATH_RESPONSE, PADDED"


In [ ]:
import numpy as np
import csv
import pprint
import pandas as pd

# --- New constant for realism ---
SIMULATED_MTU = 1350 # A conservative MTU size for path probing

def generate_realistic_features(blueprint: dict) -> list:
    """
    Generates a more realistic sequence of low-level packet feature vectors,
    including bidirectional validation, MTU padding, and protocol chattiness.
    """
    # --- 1. STATE INITIALIZATION ---
    output_packet_features = []
    current_time_msec = 0.0
    client_bytes_sent = 0
    server_bytes_sent = 0
    has_migrated = False
    time_of_last_packet = 0.0

    # --- Helper function with more frame types ---
    def create_low_level_feature(delta_time, length, direction, quic_type, frame_flags=None):
        if frame_flags is None: frame_flags = {}
        feature_vector = {
            'delta_time': round(delta_time, 4), 'packet_length': int(length), 'packet_direction': direction,
            'header_form': 1 if quic_type in ['is_initial', 'is_handshake', 'is_0rtt', 'is_retry'] else 0,
            'quic_packet_type_is_initial': 0, 'quic_packet_type_is_handshake': 0, 'quic_packet_type_is_0rtt': 0,
            'quic_packet_type_is_1rtt': 0, 'quic_packet_type_is_retry': 0, 'quic_packet_type_is_vn': 0,
            'has_path_challenge': 0, 'has_path_response': 0, 'has_new_connection_id': 0,
            'has_retire_cid': 0, 'has_padding': 0, 'has_ack': 0, 'has_connection_close': 0,
            'http3_stream_count': 0, 'http3_fin_count': 0, 'has_ping': 0 # Added PING
        }
        feature_vector[f'quic_packet_type_{quic_type}'] = 1
        feature_vector.update(frame_flags)
        return feature_vector

    # --- 2. HANDSHAKE PHASE --- (Same as before)
    handshake_duration = blueprint.get('handshake_duration_msec', 100.0) or 100.0
    if blueprint.get('retry_occurred', 0) == 1:
        delta_per_pkt = handshake_duration / 6
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 1232, 0, 'is_initial'))
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 96, 1, 'is_retry'))
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 1232, 0, 'is_initial'))
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 1232, 1, 'is_handshake'))
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 200, 0, 'is_handshake', {'has_ack': 1}))
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 96, 1, 'is_1rtt', {'has_ack': 1}))
    else:
        delta_per_pkt = handshake_duration / 4
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 1232, 0, 'is_initial'))
        cid_count = blueprint.get('server_issued_cid_count', 0)
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 1232, 1, 'is_handshake', {'has_new_connection_id': cid_count} if cid_count > 0 else {}))
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 200, 0, 'is_handshake', {'has_ack': 1}))
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 96, 1, 'is_1rtt', {'has_ack': 1}))
    current_time_msec = handshake_duration
    time_of_last_packet = handshake_duration

    # --- 3. DATA TRANSFER & MIGRATION (IMPROVED EVENT-BASED LOOP) ---
    time_to_migrate = blueprint.get('time_to_migration_msec', 0)
    if blueprint.get('migration_type') == 'NO_MIGRATION':
        time_to_migrate = blueprint['connection_duration_msec'] + 1 
    
    while current_time_msec < blueprint['connection_duration_msec']:
        made_progress_this_cycle = False
        
        # PRIORITY 1: MIGRATION EVENT
        if not has_migrated and current_time_msec >= time_to_migrate:
            has_migrated = True
            made_progress_this_cycle = True
            validation_rtt = blueprint.get('migration_validation_duration_msec', 50.0) or 50.0
            
            # --- FULL BIDIRECTIONAL VALIDATION ---
            # 1. Client sends padded PATH_CHALLENGE
            output_packet_features.append(create_low_level_feature(2.0, SIMULATED_MTU, 0, 'is_1rtt', {'has_path_challenge': 1, 'has_padding': 1}))
            current_time_msec += 2.0; time_of_last_packet = current_time_msec
            
            # 2. Server receives it and immediately sends its own padded PATH_CHALLENGE back
            output_packet_features.append(create_low_level_feature(2.0, SIMULATED_MTU, 1, 'is_1rtt', {'has_path_challenge': 1, 'has_padding': 1}))
            current_time_msec += 2.0; time_of_last_packet = current_time_msec

            # 3. Client responds to server's challenge quickly
            output_packet_features.append(create_low_level_feature(2.0, 75, 0, 'is_1rtt', {'has_path_response': 1}))
            current_time_msec += 2.0; time_of_last_packet = current_time_msec

            # 4. Server responds to client's original challenge after the RTT
            output_packet_features.append(create_low_level_feature(validation_rtt, 75, 1, 'is_1rtt', {'has_path_response': 1}))
            current_time_msec += validation_rtt; time_of_last_packet = current_time_msec
            
            continue

        # PRIORITY 2: CLIENT DATA TRANSFER
        if client_bytes_sent < blueprint['total_client_app_bytes']:
            made_progress_this_cycle = True
            avg_req_size = blueprint.get('avg_request_size', 1024) or 1024
            chunk_size = max(1, int(np.random.normal(avg_req_size, avg_req_size * 0.2)))
            output_packet_features.append(create_low_level_feature(10.0, chunk_size + 40, 0, 'is_1rtt', {'http3_stream_count': 1}))
            current_time_msec += 10.0; time_of_last_packet = current_time_msec
            client_bytes_sent += chunk_size
        
        # PRIORITY 3: SERVER DATA TRANSFER
        elif server_bytes_sent < blueprint['total_server_app_bytes']:
            made_progress_this_cycle = True
            avg_res_size = blueprint.get('avg_response_size', 1350) or 1350
            chunk_size = max(1, int(np.random.normal(avg_res_size, avg_res_size * 0.2)))
            output_packet_features.append(create_low_level_feature(10.0, chunk_size + 40, 1, 'is_1rtt', {'http3_stream_count': 1}))
            current_time_msec += 10.0; time_of_last_packet = current_time_msec
            server_bytes_sent += chunk_size

        # PRIORITY 4: PROTOCOL CHATTINESS (if nothing else happened)
        if not made_progress_this_cycle:
            idle_time = current_time_msec - time_of_last_packet
            if idle_time > 25.0: # Threshold to send a keep-alive
                # Randomly send an ACK or a PING
                if np.random.rand() > 0.5:
                    output_packet_features.append(create_low_level_feature(2.0, 65, np.random.choice([0,1]), 'is_1rtt', {'has_ack': 1}))
                else:
                    output_packet_features.append(create_low_level_feature(2.0, 68, np.random.choice([0,1]), 'is_1rtt', {'has_ping': 1}))
                current_time_msec += 2.0; time_of_last_packet = current_time_msec
            else:
                # If truly idle, just advance time to avoid infinite loop
                current_time_msec += 50.0

    # --- 4. TEARDOWN PHASE ---
    close_type = blueprint.get('connection_close_type', 'IDLE_TIMEOUT')
    if close_type != 'IDLE_TIMEOUT':
        close_direction = 1 if close_type == 'SERVER_CLOSE' else 0
        output_packet_features.append(create_low_level_feature(50.0, 62, close_direction, 'is_1rtt', {'has_connection_close': 1}))
    
    return output_packet_features

# --- EXAMPLE USAGE with PANDAS ---
if __name__ == '__main__':
    real_world_blueprint = {
        "connection_duration_msec": 107.25, "retry_occurred": 1, "server_issued_cid_count": 1,
        "migration_type": "IP_AND_PORT", "connection_close_type": "CLIENT_CLOSE",
        "handshake_duration_msec": 77.66, "total_client_app_bytes": 119,
        "total_server_app_bytes": 355, "avg_request_size": 23.8, "avg_response_size": 71.0,
        "client_bidi_streams_count": 1, "client_uni_streams_count": 4, "server_uni_streams_count": 4,
        "time_to_migration_msec": 86.83, "app_data_bytes_before_migration": 47,
        "migration_validation_duration_msec": 1.17
    }

    print("--- Generating features with REALISTIC script from REAL WORLD blueprint ---")
    low_level_packets = generate_realistic_features(real_world_blueprint)

    if not low_level_packets:
        print("No packets were generated.")
    else:
        df = pd.DataFrame(low_level_packets)
        def create_summary(row):
            info = ['C->S' if row['packet_direction'] == 0 else 'S->C']
            for col, name in [('quic_packet_type_is_initial', 'Initial'), ('quic_packet_type_is_handshake', 'Handshake'),
                              ('quic_packet_type_is_1rtt', '1-RTT'), ('quic_packet_type_is_retry', 'Retry')]:
                if row[col] == 1: info.append(name); break
            for col, name in [('has_new_connection_id', 'NEW_CID'), ('has_path_challenge', 'PATH_CHALLENGE'),
                              ('has_path_response', 'PATH_RESPONSE'), ('has_ack', 'ACK'),
                              ('has_connection_close', 'CLOSE'), ('has_ping', 'PING')]:
                if row[col] > 0: info.append(name)
            if row['has_padding'] == 1: info.append('PADDED')
            if row['http3_stream_count'] > 0: info.append(f"STREAM({row['http3_stream_count']})")
            return ', '.join(info)

        df['Summary'] = df.apply(create_summary, axis=1)
        print(f"\n--- Generated {len(df)} packets. Displaying key columns: ---")
        display_columns = ['delta_time', 'packet_length', 'Summary']
        with pd.option_context('display.max_rows', None, 'display.width', 1000):
             print(df[display_columns])
        
        output_filename = 'synthetic_low_level_packets_df.csv'
        df.to_csv(output_filename, index=False)
        print(f"\nSuccessfully saved the full DataFrame to '{output_filename}'")

--- Generating low-level features from REAL WORLD blueprint (Corrected Script) ---

--- Generated 11 packets. Displaying key columns: ---
    delta_time  packet_length                      Summary
0      12.9441           1232                C->S, Initial
1      12.9441             96                  S->C, Retry
2      12.9441           1232                C->S, Initial
3      12.9441           1232              S->C, Handshake
4      12.9441            200         C->S, Handshake, ACK
5      12.9441             96             S->C, 1-RTT, ACK
6      20.0000             63     C->S, 1-RTT, STREAM(1.0)
7       5.0000             68  C->S, 1-RTT, PATH_CHALLENGE
8       1.1742             68   S->C, 1-RTT, PATH_RESPONSE
9      20.0000             55     C->S, 1-RTT, STREAM(1.0)
10     50.0000             62           C->S, 1-RTT, CLOSE

Successfully saved the full DataFrame to 'synthetic_low_level_packets_df.csv'


In [10]:

# --- New constant for realism ---
SIMULATED_MTU = 1350 # A conservative MTU size for path probing

def generate_realistic_features(blueprint: dict) -> list:
    """
    Generates a more realistic sequence of low-level packet feature vectors,
    including bidirectional validation, MTU padding, and protocol chattiness.
    """
    # --- 1. STATE INITIALIZATION ---
    output_packet_features = []
    current_time_msec = 0.0
    client_bytes_sent = 0
    server_bytes_sent = 0
    has_migrated = False
    time_of_last_packet = 0.0

    # --- Helper function with more frame types ---
    def create_low_level_feature(delta_time, length, direction, quic_type, frame_flags=None):
        if frame_flags is None: frame_flags = {}
        feature_vector = {
            'delta_time': round(delta_time, 4), 'packet_length': int(length), 'packet_direction': direction,
            'header_form': 1 if quic_type in ['is_initial', 'is_handshake', 'is_0rtt', 'is_retry'] else 0,
            'quic_packet_type_is_initial': 0, 'quic_packet_type_is_handshake': 0, 'quic_packet_type_is_0rtt': 0,
            'quic_packet_type_is_1rtt': 0, 'quic_packet_type_is_retry': 0, 'quic_packet_type_is_vn': 0,
            'has_path_challenge': 0, 'has_path_response': 0, 'has_new_connection_id': 0,
            'has_retire_cid': 0, 'has_padding': 0, 'has_ack': 0, 'has_connection_close': 0,
            'http3_stream_count': 0, 'http3_fin_count': 0, 'has_ping': 0 # Added PING
        }
        feature_vector[f'quic_packet_type_{quic_type}'] = 1
        feature_vector.update(frame_flags)
        return feature_vector

    # --- 2. HANDSHAKE PHASE --- (Same as before)
    handshake_duration = blueprint.get('handshake_duration_msec', 100.0) or 100.0
    if blueprint.get('retry_occurred', 0) == 1:
        delta_per_pkt = handshake_duration / 6
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 1232, 0, 'is_initial'))
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 96, 1, 'is_retry'))
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 1232, 0, 'is_initial'))
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 1232, 1, 'is_handshake'))
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 200, 0, 'is_handshake', {'has_ack': 1}))
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 96, 1, 'is_1rtt', {'has_ack': 1}))
    else:
        delta_per_pkt = handshake_duration / 4
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 1232, 0, 'is_initial'))
        cid_count = blueprint.get('server_issued_cid_count', 0)
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 1232, 1, 'is_handshake', {'has_new_connection_id': cid_count} if cid_count > 0 else {}))
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 200, 0, 'is_handshake', {'has_ack': 1}))
        output_packet_features.append(create_low_level_feature(delta_per_pkt, 96, 1, 'is_1rtt', {'has_ack': 1}))
    current_time_msec = handshake_duration
    time_of_last_packet = handshake_duration

    # --- 3. DATA TRANSFER & MIGRATION (IMPROVED EVENT-BASED LOOP) ---
    time_to_migrate = blueprint.get('time_to_migration_msec', 0)
    if blueprint.get('migration_type') == 'NO_MIGRATION':
        time_to_migrate = blueprint['connection_duration_msec'] + 1 
    
    while current_time_msec < blueprint['connection_duration_msec']:
        made_progress_this_cycle = False
        
        # PRIORITY 1: MIGRATION EVENT
        if not has_migrated and current_time_msec >= time_to_migrate:
            has_migrated = True
            made_progress_this_cycle = True
            validation_rtt = blueprint.get('migration_validation_duration_msec', 50.0) or 50.0
            
            # --- FULL BIDIRECTIONAL VALIDATION ---
            # 1. Client sends padded PATH_CHALLENGE
            output_packet_features.append(create_low_level_feature(2.0, SIMULATED_MTU, 0, 'is_1rtt', {'has_path_challenge': 1, 'has_padding': 1}))
            current_time_msec += 2.0; time_of_last_packet = current_time_msec
            
            # 2. Server receives it and immediately sends its own padded PATH_CHALLENGE back
            output_packet_features.append(create_low_level_feature(2.0, SIMULATED_MTU, 1, 'is_1rtt', {'has_path_challenge': 1, 'has_padding': 1}))
            current_time_msec += 2.0; time_of_last_packet = current_time_msec

            # 3. Client responds to server's challenge quickly
            output_packet_features.append(create_low_level_feature(2.0, 75, 0, 'is_1rtt', {'has_path_response': 1}))
            current_time_msec += 2.0; time_of_last_packet = current_time_msec

            # 4. Server responds to client's original challenge after the RTT
            output_packet_features.append(create_low_level_feature(validation_rtt, 75, 1, 'is_1rtt', {'has_path_response': 1}))
            current_time_msec += validation_rtt; time_of_last_packet = current_time_msec
            
            continue

        # PRIORITY 2: CLIENT DATA TRANSFER
        if client_bytes_sent < blueprint['total_client_app_bytes']:
            made_progress_this_cycle = True
            avg_req_size = blueprint.get('avg_request_size', 1024) or 1024
            chunk_size = max(1, int(np.random.normal(avg_req_size, avg_req_size * 0.2)))
            output_packet_features.append(create_low_level_feature(10.0, chunk_size + 40, 0, 'is_1rtt', {'http3_stream_count': 1}))
            current_time_msec += 10.0; time_of_last_packet = current_time_msec
            client_bytes_sent += chunk_size
        
        # PRIORITY 3: SERVER DATA TRANSFER
        elif server_bytes_sent < blueprint['total_server_app_bytes']:
            made_progress_this_cycle = True
            avg_res_size = blueprint.get('avg_response_size', 1350) or 1350
            chunk_size = max(1, int(np.random.normal(avg_res_size, avg_res_size * 0.2)))
            output_packet_features.append(create_low_level_feature(10.0, chunk_size + 40, 1, 'is_1rtt', {'http3_stream_count': 1}))
            current_time_msec += 10.0; time_of_last_packet = current_time_msec
            server_bytes_sent += chunk_size

        # PRIORITY 4: PROTOCOL CHATTINESS (if nothing else happened)
        if not made_progress_this_cycle:
            idle_time = current_time_msec - time_of_last_packet
            if idle_time > 25.0: # Threshold to send a keep-alive
                # Randomly send an ACK or a PING
                if np.random.rand() > 0.5:
                    output_packet_features.append(create_low_level_feature(2.0, 65, np.random.choice([0,1]), 'is_1rtt', {'has_ack': 1}))
                else:
                    output_packet_features.append(create_low_level_feature(2.0, 68, np.random.choice([0,1]), 'is_1rtt', {'has_ping': 1}))
                current_time_msec += 2.0; time_of_last_packet = current_time_msec
            else:
                # If truly idle, just advance time to avoid infinite loop
                current_time_msec += 50.0

    # --- 4. TEARDOWN PHASE ---
    close_type = blueprint.get('connection_close_type', 'IDLE_TIMEOUT')
    if close_type != 'IDLE_TIMEOUT':
        close_direction = 1 if close_type == 'SERVER_CLOSE' else 0
        output_packet_features.append(create_low_level_feature(50.0, 62, close_direction, 'is_1rtt', {'has_connection_close': 1}))
    
    return output_packet_features



In [11]:
# This is a sample blueprint, as if it were generated by your trained GAN
sample_blueprint = {
    "connection_duration_msec": 107.25188255310059,
    "retry_occurred": 1,
    "server_issued_cid_count": 1,
    "migration_type": "IP_AND_PORT",
    "connection_close_type": "CLIENT_CLOSE",
    "handshake_duration_msec": 77.66485214233398,
    "total_client_app_bytes": 119,
    "total_server_app_bytes": 355,
    "avg_request_size": 23.8,
    "avg_response_size": 71.0,
    "client_bidi_streams_count": 1,
    "client_uni_streams_count": 4,
    "server_uni_streams_count": 4,
    "time_to_migration_msec": 86.83586120605469,
    "app_data_bytes_before_migration": 47,
    "migration_validation_duration_msec": 1.1742115020751953
}

print("--- Generating low-level features from sample blueprint ---")
low_level_packets = generate_realistic_features(sample_blueprint)

print(f"Generated a total of {len(low_level_packets)} packet feature vectors.")
print("\n--- First 5 generated packets: ---")
pprint.pprint(low_level_packets[:5])
print("\n--- Last 5 generated packets: ---")
pprint.pprint(low_level_packets[-5:])

--- Generating low-level features from sample blueprint ---
Generated a total of 14 packet feature vectors.

--- First 5 generated packets: ---
[{'delta_time': 12.9441,
  'has_ack': 0,
  'has_connection_close': 0,
  'has_new_connection_id': 0,
  'has_padding': 0,
  'has_path_challenge': 0,
  'has_path_response': 0,
  'has_ping': 0,
  'has_retire_cid': 0,
  'header_form': 1,
  'http3_fin_count': 0,
  'http3_stream_count': 0,
  'packet_direction': 0,
  'packet_length': 1232,
  'quic_packet_type_is_0rtt': 0,
  'quic_packet_type_is_1rtt': 0,
  'quic_packet_type_is_handshake': 0,
  'quic_packet_type_is_initial': 1,
  'quic_packet_type_is_retry': 0,
  'quic_packet_type_is_vn': 0},
 {'delta_time': 12.9441,
  'has_ack': 0,
  'has_connection_close': 0,
  'has_new_connection_id': 0,
  'has_padding': 0,
  'has_path_challenge': 0,
  'has_path_response': 0,
  'has_ping': 0,
  'has_retire_cid': 0,
  'header_form': 1,
  'http3_fin_count': 0,
  'http3_stream_count': 0,
  'packet_direction': 1,
  'pack

In [14]:
import pandas as pd
if not low_level_packets:
        print("No packets were generated.")
else:
    # --- 2. Create the DataFrame ---
    df = pd.DataFrame(low_level_packets)

    # --- 3. Create a human-readable Summary column ---
    def create_summary(row):
        info = []
        # Direction
        info.append('C->S' if row['packet_direction'] == 0 else 'S->C')
        # Packet Type
        for col, name in [('quic_packet_type_is_initial', 'Initial'), ('quic_packet_type_is_handshake', 'Handshake'),
                            ('quic_packet_type_is_1rtt', '1-RTT'), ('quic_packet_type_is_retry', 'Retry')]:
            if row[col] == 1:
                info.append(name)
                break
        # Important Frames
        for col, name in [('has_new_connection_id', 'NEW_CID'), ('has_path_challenge', 'PATH_CHALLENGE'),
                            ('has_path_response', 'PATH_RESPONSE'), ('has_ack', 'ACK'),
                            ('has_connection_close', 'CLOSE')]:
            if row[col] > 0:
                info.append(name)
        if row['http3_stream_count'] > 0:
            info.append(f"STREAM({row['http3_stream_count']})")
        return ', '.join(info)

    df['Summary'] = df.apply(create_summary, axis=1)

    # --- 4. Display the DataFrame ---
    print(f"\n--- Generated {len(df)} packets. Displaying key columns: ---")
    
    # Select the most important columns for a clean display
    display_columns = ['delta_time', 'packet_length', 'Summary']
    with pd.option_context('display.max_rows', None, 'display.width', 1000):
            print(df[display_columns])
        
    display(df)


--- Generated 14 packets. Displaying key columns: ---
    delta_time  packet_length                      Summary
0      12.9441           1232                C->S, Initial
1      12.9441             96                  S->C, Retry
2      12.9441           1232                C->S, Initial
3      12.9441           1232              S->C, Handshake
4      12.9441            200         C->S, Handshake, ACK
5      12.9441             96             S->C, 1-RTT, ACK
6      10.0000             62     C->S, 1-RTT, STREAM(1.0)
7       2.0000           1350  C->S, 1-RTT, PATH_CHALLENGE
8       2.0000           1350  S->C, 1-RTT, PATH_CHALLENGE
9       2.0000             75   C->S, 1-RTT, PATH_RESPONSE
10      1.1742             75   S->C, 1-RTT, PATH_RESPONSE
11     10.0000             63     C->S, 1-RTT, STREAM(1.0)
12     10.0000             56     C->S, 1-RTT, STREAM(1.0)
13     50.0000             62           C->S, 1-RTT, CLOSE


,delta_time,packet_length,packet_direction,header_form,quic_packet_type_is_initial,quic_packet_type_is_handshake,quic_packet_type_is_0rtt,quic_packet_type_is_1rtt,quic_packet_type_is_retry,quic_packet_type_is_vn,has_path_challenge,has_path_response,has_new_connection_id,has_retire_cid,has_padding,has_ack,has_connection_close,http3_stream_count,http3_fin_count,has_ping,Summary
0,12.9441,1232,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"C->S, Initial"
1,12.9441,96,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,"S->C, Retry"
2,12.9441,1232,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"C->S, Initial"
3,12.9441,1232,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"S->C, Handshake"
4,12.9441,200,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,"C->S, Handshake, ACK"
5,12.9441,96,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,"S->C, 1-RTT, ACK"
6,10.0000,62,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,"C->S, 1-RTT, STREAM(1.0)"
7,2.0000,1350,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,"C->S, 1-RTT, PATH_CHALLENGE"
8,2.0000,1350,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,"S->C, 1-RTT, PATH_CHALLENGE"
9,2.0000,75,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,"C->S, 1-RTT, PATH_RESPONSE"


In [13]:
pd.set_option('display.max_columns', None)  # Show all columns when printing

In [ ]:
# --- How to save the output to a CSV file ---
output_filename = 'synthetic_low_level_packets.csv'
if low_level_packets:
    # Get the headers from the first packet's keys
    headers = low_level_packets[0].keys()
    
    with open(output_filename, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=headers)
        writer.writeheader()
        writer.writerows(low_level_packets)
    
    print(f"\nSuccessfully saved the full sequence to '{output_filename}'")

In [65]:
import numpy as np
import csv
import pprint
import pandas as pd

SIMULATED_MTU = 1350
PATH_VALIDATION_MTU = 1398  # MTU for PATH_CHALLENGE/RESPONSE frames (RFC requirement)
QUIC_OVERHEAD = 40  # Approximate overhead per QUIC packet
HTTP3_FRAME_OVERHEAD = 10  # HTTP/3 frame header overhead

def generate_statistically_realistic_features(blueprint: dict, stats: dict) -> list:
    """
    Generates a statistically realistic sequence of low-level packet features,
    matching the actual behavior seen in QUIC connection migration captures.
    
    Pattern:
    1. Handshake (with optional retry)
    2. HTTP/3 SETTINGS exchange (multiple small unidirectional streams)
    3. Small bidirectional request/response streams
    4. Connection migration (PATH_CHALLENGE/RESPONSE with MTU-sized packets)
    5. Connection close
    """
    output_packet_features = []
    current_time_msec = 0.0
    current_packet_count = 0
    
    # Track application bytes sent
    client_app_bytes_sent = 0
    server_app_bytes_sent = 0
    
    # Migration state
    has_migrated = False
    
    # Helper to safely draw from stats
    def get_stat(category, key, fallback_mean=0, fallback_std=0):
        if category in stats and key in stats[category] and stats[category][key]['samples'] > 0:
            mean = stats[category][key]['mean']
            std = stats[category][key]['std']
            value = np.random.normal(mean, std)
            # Ensure reasonable bounds
            if 'delta' in key or 'time' in key:
                return max(0.00001, value)  # Delta times should be positive
            elif 'size' in key or 'length' in key:
                return max(50, min(SIMULATED_MTU, value))  # Reasonable packet sizes
            return max(1, value)
        return max(1, np.random.normal(fallback_mean, fallback_std))

    def create_low_level_feature(delta_time, length, direction, header_form, 
                                  count_initial=0, count_0rtt=0, count_handshake=0, 
                                  count_1rtt=0, count_retry=0, count_vn=0,
                                  count_ack=0, count_padding=0, count_connection_close=0,
                                  count_path_challenge=0, count_path_response=0,
                                  count_new_connection_id=0, count_retire_cid=0,
                                  count_crypto=0, count_handshake_done=0,
                                  http3_stream_count=0, http3_fin_count=0,
                                  stream_length=0, stream_type_count=0):
        nonlocal current_packet_count, current_time_msec
        current_packet_count += 1
        current_time_msec += delta_time
        
        return {
            'frame_number': current_packet_count,
            'delta_time': round(delta_time, 6),
            'packet_length': int(length),
            'packet_direction': direction,
            'header_form': header_form,
            'count_initial': count_initial,
            'count_0rtt': count_0rtt,
            'count_handshake': count_handshake,
            'count_1rtt': count_1rtt,
            'count_retry': count_retry,
            'count_vn': count_vn,
            'count_ack': count_ack,
            'count_padding': count_padding,
            'count_connection_close': count_connection_close,
            'count_path_challenge': count_path_challenge,
            'count_path_response': count_path_response,
            'count_new_connection_id': count_new_connection_id,
            'count_retire_cid': count_retire_cid,
            'count_crypto': count_crypto,
            'count_handshake_done': count_handshake_done,
            'http3_stream_count': http3_stream_count,
            'http3_fin_count': http3_fin_count,
            'stream_length': stream_length,
            'stream_type_count': stream_type_count
        }

    # =================================================================
    # PHASE 1: HANDSHAKE
    # =================================================================
    if blueprint.get('retry_occurred', 0) == 1:
        # Packet 1: Client Initial (1-RTT, long header)
        delta = 0.0
        size = get_stat('packet_sizes', 'handshake_initial_client', 1248, 20)
        output_packet_features.append(create_low_level_feature(
            delta, size, 0, header_form=0, count_1rtt=1
        ))
        
        # Packet 2: Server VN or early response
        delta = get_stat('delta_times', 'handshake_s2c', 0.0002, 0.0001)
        size = 95
        output_packet_features.append(create_low_level_feature(
            delta, size, 1, header_form=0, count_1rtt=1, count_vn=1
        ))
        
        # Packet 3: Client Initial with retry token (long header)
        delta = get_stat('delta_times', 'handshake_c2s', 0.0008, 0.0002)
        size = get_stat('packet_sizes', 'handshake_initial_client', 1248, 20)
        output_packet_features.append(create_low_level_feature(
            delta, size, 0, header_form=1, count_initial=1, count_crypto=1
        ))
        
        # Packet 4: Server Retry
        delta = get_stat('delta_times', 'handshake_s2c', 0.0001, 0.00005)
        size = 137
        output_packet_features.append(create_low_level_feature(
            delta, size, 1, header_form=1, count_retry=1
        ))
        
        # Packet 5: New Client Initial
        delta = get_stat('delta_times', 'handshake_c2s', 0.0003, 0.0001)
        size = get_stat('packet_sizes', 'handshake_initial_client', 1248, 20)
        output_packet_features.append(create_low_level_feature(
            delta, size, 0, header_form=1, count_initial=1, count_crypto=1
        ))
        
        # Packet 6: Server Initial + Handshake
        delta = get_stat('delta_times', 'handshake_s2c', 0.0015, 0.0005)
        size = 1248
        output_packet_features.append(create_low_level_feature(
            delta, size, 1, header_form=1, count_initial=1, count_handshake=1, 
            count_ack=1, count_crypto=2
        ))
        
        # Packet 7: Server Handshake continuation
        delta = get_stat('delta_times', 'handshake_s2c', 0.00007, 0.00002)
        size = 517
        output_packet_features.append(create_low_level_feature(
            delta, size, 1, header_form=1, count_handshake=1, count_crypto=1
        ))
        
        # Packet 8: Client Handshake + NEW_CONNECTION_ID
        delta = get_stat('delta_times', 'handshake_c2s', 0.0035, 0.001)
        size = 1398
        output_packet_features.append(create_low_level_feature(
            delta, size, 0, header_form=1, count_initial=1, count_handshake=1, count_1rtt=1,
            count_ack=2, count_padding=1, count_new_connection_id=1, count_crypto=1
        ))
        
    else:
        # Standard handshake without retry (simplified 4-packet version)
        # Packet 1: Client Initial
        delta = 0.0
        size = get_stat('packet_sizes', 'handshake_initial_client', 1248, 20)
        output_packet_features.append(create_low_level_feature(
            delta, size, 0, header_form=1, count_initial=1, count_crypto=1
        ))
        
        # Packet 2: Server Initial + Handshake
        delta = get_stat('delta_times', 'handshake_s2c', 0.020, 0.010)
        size = 1248
        output_packet_features.append(create_low_level_feature(
            delta, size, 1, header_form=1, count_initial=1, count_handshake=1,
            count_ack=1, count_crypto=2
        ))
        
        # Packet 3: Client Handshake
        delta = get_stat('delta_times', 'handshake_c2s', 0.025, 0.010)
        size = 1398
        output_packet_features.append(create_low_level_feature(
            delta, size, 0, header_form=1, count_initial=1, count_handshake=1, count_1rtt=1,
            count_ack=2, count_padding=1, count_new_connection_id=1, count_crypto=1
        ))

    # =================================================================
    # PHASE 2: HTTP/3 INITIALIZATION (SETTINGS + Control Streams)
    # =================================================================
    # Server sends HANDSHAKE_DONE + NEW_CONNECTION_ID + HTTP/3 SETTINGS
    delta = get_stat('delta_times', 'server_response', 0.0005, 0.0002)
    size = 556
    output_packet_features.append(create_low_level_feature(
        delta, size, 1, header_form=0, count_1rtt=1, count_ack=1,
        count_new_connection_id=1, count_crypto=1, count_handshake_done=1,
        http3_stream_count=1, stream_length=19, stream_type_count=1
    ))
    
    # Server sends multiple unidirectional control streams (SETTINGS, QPACK, etc.)
    server_uni_count = blueprint.get('server_uni_streams_count', 4)
    for i in range(min(server_uni_count - 1, 3)):  # Already sent 1 above
        delta = get_stat('delta_times', 'server_response', 0.00003, 0.00001)
        size = 92
        output_packet_features.append(create_low_level_feature(
            delta, size, 1, header_form=0, count_1rtt=1,
            http3_stream_count=1, stream_length=1, stream_type_count=1
        ))
        server_app_bytes_sent += 1
    
    # Server might send one more with FIN
    if server_uni_count >= 4:
        delta = get_stat('delta_times', 'server_response', 0.00002, 0.00001)
        size = 117
        output_packet_features.append(create_low_level_feature(
            delta, size, 1, header_form=0, count_1rtt=1,
            http3_stream_count=1, http3_fin_count=1, stream_length=26, stream_type_count=1
        ))
        server_app_bytes_sent += 26

    # Client ACK
    delta = get_stat('delta_times', 'ack_response', 0.0005, 0.0002)
    size = 91
    output_packet_features.append(create_low_level_feature(
        delta, size, 0, header_form=0, count_1rtt=1, count_ack=1
    ))

    # =================================================================
    # PHASE 3: CLIENT REQUESTS (Small bidirectional/unidirectional streams)
    # =================================================================
    # Client sends PATH_CHALLENGE (pre-migration probing) + unidirectional streams
    delta = get_stat('delta_times', 'client_request', 0.0002, 0.0001)
    size = 1398
    output_packet_features.append(create_low_level_feature(
        delta, size, 0, header_form=0, count_1rtt=1, count_padding=1,
        count_path_challenge=1
    ))
    
    # Server PATH_RESPONSE
    delta = get_stat('delta_times', 'server_response', 0.0005, 0.0002)
    size = 1441
    output_packet_features.append(create_low_level_feature(
        delta, size, 1, header_form=0, count_1rtt=1
    ))
    
    # Client ACK with PATH_RESPONSE
    delta = get_stat('delta_times', 'ack_response', 0.0003, 0.0001)
    size = 100
    output_packet_features.append(create_low_level_feature(
        delta, size, 0, header_form=0, count_1rtt=1, count_ack=1, count_path_response=1
    ))
    
    # Server ACK
    delta = get_stat('delta_times', 'ack_response', 0.0003, 0.0001)
    size = 91
    output_packet_features.append(create_low_level_feature(
        delta, size, 1, header_form=0, count_1rtt=1, count_ack=1
    ))

    # Client sends unidirectional streams (SETTINGS, QPACK encoders, etc.)
    client_uni_count = blueprint.get('client_uni_streams_count', 4)
    for i in range(client_uni_count):
        delta = get_stat('delta_times', 'client_request', 0.0003, 0.0001)
        
        if i == 0:  # First stream with SETTINGS
            size = 110
            stream_len = 19
        elif i < client_uni_count - 2:  # Middle streams
            size = 92
            stream_len = 1
        elif i == client_uni_count - 2:  # Second to last
            size = 163
            stream_len = 72
        else:  # Last stream with FIN
            size = 117
            stream_len = 26
            
        output_packet_features.append(create_low_level_feature(
            delta, size, 0, header_form=0, count_1rtt=1,
            http3_stream_count=1, 
            http3_fin_count=1 if i >= client_uni_count - 2 else 0,
            stream_length=stream_len, 
            stream_type_count=1
        ))
        client_app_bytes_sent += stream_len

    # =================================================================
    # PHASE 4: BIDIRECTIONAL REQUEST/RESPONSE
    # =================================================================
    # Client sends bidirectional request (e.g., GET request)
    client_bidi_count = blueprint.get('client_bidi_streams_count', 1)
    for _ in range(client_bidi_count):
        delta = get_stat('delta_times', 'client_request', 0.0015, 0.0005)
        size = 211
        request_bytes = blueprint.get('avg_request_size', 115)
        output_packet_features.append(create_low_level_feature(
            delta, size, 1, header_form=0, count_1rtt=1, count_ack=1,
            http3_stream_count=1, http3_fin_count=1, 
            stream_length=int(request_bytes), stream_type_count=1
        ))
        client_app_bytes_sent += request_bytes
        server_app_bytes_sent += 1  # Small server data in piggyback

    # =================================================================
    # PHASE 5: CONNECTION MIGRATION (RFC-compliant)
    # =================================================================
    migration_type = blueprint.get('migration_type', 'IP_AND_PORT')
    if migration_type != 'NONE' and not has_migrated:
        has_migrated = True
        
        # Wait until time_to_migration
        time_until_migration = blueprint.get('time_to_migration_msec', 50.0) - current_time_msec
        if time_until_migration > 0:
            # Idle period or small keep-alive
            delta = time_until_migration / 1000.0  # Convert to seconds
            delta = max(0.001, delta)
        else:
            delta = 0.001
        
        # Client initiates migration with PATH_CHALLENGE on new path
        # RFC 9000: PATH_CHALLENGE frames SHOULD be padded to at least 1200 bytes
        size = PATH_VALIDATION_MTU
        output_packet_features.append(create_low_level_feature(
            delta, size, 0, header_form=0, count_1rtt=1,
            count_path_challenge=1, count_padding=1
        ))
        
        # Server responds with PATH_RESPONSE + PATH_CHALLENGE
        # RFC 9000: PATH_RESPONSE packets SHOULD also be padded
        validation_rtt = blueprint.get('migration_validation_duration_msec', 1.0) / 1000.0
        delta = validation_rtt / 2.0  # Server responds partway through RTT
        size = PATH_VALIDATION_MTU
        output_packet_features.append(create_low_level_feature(
            delta, size, 1, header_form=0, count_1rtt=1,
            count_path_response=1, count_path_challenge=1, count_padding=1
        ))
        
        # Client responds with PATH_RESPONSE + ACK
        delta = validation_rtt / 2.0
        size = PATH_VALIDATION_MTU
        output_packet_features.append(create_low_level_feature(
            delta, size, 0, header_form=0, count_1rtt=1, count_ack=1,
            count_path_response=1, count_padding=1
        ))
        
        # Server ACKs the validation
        delta = get_stat('delta_times', 'ack_response', 0.0003, 0.0001)
        size = 91
        output_packet_features.append(create_low_level_feature(
            delta, size, 1, header_form=0, count_1rtt=1, count_ack=1
        ))

    # =================================================================
    # PHASE 6: POST-MIGRATION DATA (if any remaining)
    # =================================================================
    # Send any remaining application data after migration
    total_client_bytes = blueprint.get('total_client_app_bytes', 0)
    total_server_bytes = blueprint.get('total_server_app_bytes', 0)
    
    # Typically minimal post-migration data in small connections
    remaining_server_bytes = total_server_bytes - server_app_bytes_sent
    if remaining_server_bytes > 0:
        # Server sends small data packets
        num_packets = max(1, int(remaining_server_bytes / 50))
        bytes_per_packet = remaining_server_bytes / num_packets
        
        for i in range(num_packets):
            delta = get_stat('delta_times', 'server_response', 0.0002, 0.0001)
            size = int(bytes_per_packet + QUIC_OVERHEAD + HTTP3_FRAME_OVERHEAD)
            size = min(size, 200)  # Keep packets small
            
            output_packet_features.append(create_low_level_feature(
                delta, size, 1, header_form=0, count_1rtt=1,
                http3_stream_count=1 if i == num_packets - 1 else 0,
                stream_length=int(bytes_per_packet), stream_type_count=1
            ))
            server_app_bytes_sent += bytes_per_packet

    # =================================================================
    # PHASE 7: CONNECTION CLOSE (RFC-compliant)
    # =================================================================
    close_type = blueprint.get('connection_close_type', 'CLIENT_CLOSE')
    
    # Wait until connection duration
    time_until_close = blueprint.get('connection_duration_msec', 100.0) - current_time_msec
    if time_until_close > 0.010:
        delta = time_until_close / 1000.0
    else:
        delta = 0.010
    
    if close_type == 'CLIENT_CLOSE':
        # Client sends CONNECTION_CLOSE
        size = 97
        output_packet_features.append(create_low_level_feature(
            delta, size, 0, header_form=0, count_1rtt=1, count_connection_close=1
        ))
        
        # Server ACKs (optional, may be dropped in real traces)
        delta = get_stat('delta_times', 'ack_response', 0.0005, 0.0002)
        size = 91
        output_packet_features.append(create_low_level_feature(
            delta, size, 1, header_form=0, count_1rtt=1, count_ack=1
        ))
    
    elif close_type == 'SERVER_CLOSE':
        # Server sends CONNECTION_CLOSE
        size = 97
        output_packet_features.append(create_low_level_feature(
            delta, size, 1, header_form=0, count_1rtt=1, count_connection_close=1
        ))
        
        # Client ACKs
        delta = get_stat('delta_times', 'ack_response', 0.0005, 0.0002)
        size = 91
        output_packet_features.append(create_low_level_feature(
            delta, size, 0, header_form=0, count_1rtt=1, count_ack=1
        ))
    # else: IDLE_TIMEOUT - no explicit close frame

    return output_packet_features


In [81]:
import numpy as np
import csv
import pandas as pd
from typing import Dict, List, Tuple, Optional

SIMULATED_MTU = 1350
PATH_VALIDATION_MTU_MIN = 1200  # RFC 9000 minimum for path validation
PATH_VALIDATION_MTU_MAX = 1450  # Allow some variance


def generate_statistically_realistic_features(blueprint: dict, stats: dict) -> list:
    """
    Generates a flexible, statistics-driven sequence of QUIC packets.
    
    This version:
    - Uses statistics for ALL packet sizes (not hardcoded)
    - Adds realistic randomness to delta times based on network conditions
    - Adapts to the high-level blueprint while maintaining protocol correctness
    - Generates varied captures from the same blueprint
    """
    output_packet_features = []
    current_time_msec = 0.0
    current_packet_count = 0
    
    # Track application bytes
    client_app_bytes_sent = 0
    server_app_bytes_sent = 0
    has_migrated = False
    
    # Network jitter simulation (adds realism to delta times)
    base_rtt = blueprint.get('migration_validation_duration_msec', 20.0) / 2.0  # Half RTT
    network_jitter = np.random.uniform(0.5, 1.5)  # Random network conditions
    
    def get_stat(category: str, key: str, fallback_mean: float = 100, 
                 fallback_std: float = 20, strict_positive: bool = False) -> float:
        """
        Safely draw from statistics with fallbacks and bounds.
        """
        if category in stats and key in stats[category] and stats[category][key]['samples'] > 0:
            mean = stats[category][key]['mean']
            std = stats[category][key]['std']
            value = np.random.normal(mean, std)
        else:
            value = np.random.normal(fallback_mean, fallback_std)
        
        # Apply bounds
        if 'delta' in key or 'time' in key:
            return max(0.000001, value) * network_jitter
        elif 'size' in key or 'length' in key:
            min_val = 50 if strict_positive else 40
            return int(max(min_val, min(SIMULATED_MTU, value)))
        elif strict_positive:
            return max(1, value)
        return value
    
    def get_delta_time(phase: str, direction: str = 'both') -> float:
        """
        Get realistic delta time based on phase and direction.
        Adds randomness for network conditions.
        """
        if phase == 'handshake':
            if direction == 'c2s':
                base = get_stat('delta_times', 'handshake_c2s', 0.015, 0.010)
            else:
                base = get_stat('delta_times', 'handshake_s2c', 0.020, 0.015)
        elif phase == 'data':
            if direction == 'c2s':
                base = get_stat('delta_times', 'client_request', 0.010, 0.005)
            else:
                base = get_stat('delta_times', 'server_response', 0.005, 0.003)
        elif phase == 'ack':
            base = get_stat('delta_times', 'ack_response', 0.002, 0.001)
        else:
            base = 0.001
        
        # Add random jitter (10-50% variance)
        jitter = np.random.uniform(0.7, 1.3)
        return max(0.000001, base * jitter)
    
    def get_packet_size(packet_type: str, fallback_mean: int = 100, 
                       fallback_std: int = 20) -> int:
        """
        Get packet size from statistics with protocol-aware fallbacks.
        """
        # Try to get from stats first
        size_key = f'{packet_type}'
        size = get_stat('packet_sizes', size_key, fallback_mean, fallback_std)
        
        # Ensure minimum sizes for specific packet types
        if 'initial' in packet_type.lower():
            return max(1200, int(size))  # Initial packets need to be large
        elif 'path_' in packet_type.lower():
            return int(np.random.uniform(PATH_VALIDATION_MTU_MIN, PATH_VALIDATION_MTU_MAX))
        elif 'ack' in packet_type.lower():
            return max(60, min(150, int(size)))  # ACKs are small
        
        return int(size)
    
    def create_packet(delta: float, length: int, direction: int, header_form: int,
                     **counts) -> dict:
        """
        Create a packet feature dictionary with all fields.
        """
        nonlocal current_packet_count, current_time_msec
        current_packet_count += 1
        current_time_msec += delta
        
        # Default all counts to 0
        packet = {
            'frame_number': current_packet_count,
            'delta_time': round(delta, 6),
            'packet_length': int(length),
            'packet_direction': direction,
            'header_form': header_form,
            'count_initial': 0,
            'count_0rtt': 0,
            'count_handshake': 0,
            'count_1rtt': 0,
            'count_retry': 0,
            'count_vn': 0,
            'count_ack': 0,
            'count_padding': 0,
            'count_connection_close': 0,
            'count_path_challenge': 0,
            'count_path_response': 0,
            'count_new_connection_id': 0,
            'count_retire_cid': 0,
            'count_crypto': 0,
            'count_handshake_done': 0,
            'http3_stream_count': 0,
            'http3_fin_count': 0,
            'stream_length': 0,
            'stream_type_count': 0
        }
        
        # Update with provided counts
        packet.update(counts)
        return packet
    
    # =================================================================
    # PHASE 1: HANDSHAKE
    # =================================================================
    if blueprint.get('retry_occurred', 0) == 1:
        # Retry handshake sequence
        # 1. Client Initial attempt
        output_packet_features.append(create_packet(
            0.0,
            get_packet_size('handshake_initial_client', 1248, 30),
            0, 0, count_1rtt=1
        ))
        
        # 2. Server early response (VN or similar)
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 's2c'),
            get_packet_size('ack_server', 95, 15),
            1, 0, count_1rtt=1, count_vn=1
        ))
        
        # 3. Client Initial (after seeing VN)
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 'c2s'),
            get_packet_size('handshake_initial_client', 1248, 30),
            0, 1, count_initial=1, count_crypto=1
        ))
        
        # 4. Server Retry
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 's2c'),
            get_packet_size('handshake_other_server', 137, 20),
            1, 1, count_retry=1
        ))
        
        # 5. Client Initial with retry token
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 'c2s'),
            get_packet_size('handshake_initial_client', 1248, 30),
            0, 1, count_initial=1, count_crypto=1
        ))
        
        # 6. Server Initial + Handshake (large packet)
        num_crypto = np.random.randint(1, 3)  # Variable crypto frames
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 's2c'),
            get_packet_size('handshake_initial_client', 1248, 50),
            1, 1, 
            count_initial=1, 
            count_handshake=np.random.randint(0, 2),
            count_ack=1, 
            count_crypto=num_crypto
        ))
        
        # 7. Server Handshake continuation (if needed)
        if np.random.random() > 0.3:  # 70% chance of continuation packet
            output_packet_features.append(create_packet(
                get_delta_time('handshake', 's2c'),
                get_packet_size('handshake_other_server', 517, 100),
                1, 1, count_handshake=1, count_crypto=1
            ))
        
        # 8. Client Handshake completion
        num_packet_types = np.random.randint(2, 4)  # Variable packet type mixing
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 'c2s'),
            get_packet_size('handshake_other_client', 1398, 100),
            0, 1,
            count_initial=np.random.randint(0, 2),
            count_handshake=1,
            count_1rtt=np.random.randint(0, 2),
            count_ack=np.random.randint(1, 3),
            count_padding=np.random.randint(0, 2),
            count_new_connection_id=1,
            count_crypto=1
        ))
    else:
        # Standard handshake (no retry)
        # 1. Client Initial
        output_packet_features.append(create_packet(
            0.0,
            get_packet_size('handshake_initial_client', 1248, 30),
            0, 1, count_initial=1, count_crypto=1
        ))
        
        # 2. Server Initial + Handshake
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 's2c'),
            get_packet_size('handshake_initial_client', 1248, 50),
            1, 1,
            count_initial=1,
            count_handshake=np.random.randint(0, 2),
            count_ack=1,
            count_crypto=np.random.randint(1, 3)
        ))
        
        # 3. Possible Server Handshake continuation
        if np.random.random() > 0.4:
            output_packet_features.append(create_packet(
                get_delta_time('handshake', 's2c'),
                get_packet_size('handshake_other_server', 517, 100),
                1, 1, count_handshake=1, count_crypto=1
            ))
        
        # 4. Client Handshake completion
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 'c2s'),
            get_packet_size('handshake_other_client', 1398, 100),
            0, 1,
            count_initial=np.random.randint(0, 2),
            count_handshake=1,
            count_1rtt=np.random.randint(0, 2),
            count_ack=np.random.randint(1, 3),
            count_padding=np.random.randint(0, 2),
            count_new_connection_id=1,
            count_crypto=np.random.randint(0, 2)
        ))
    
    # =================================================================
    # PHASE 2: HTTP/3 INITIALIZATION
    # =================================================================
    # Server sends HANDSHAKE_DONE + initial HTTP/3 setup
    settings_size = np.random.randint(15, 25)  # Variable SETTINGS size
    output_packet_features.append(create_packet(
        get_delta_time('data', 's2c'),
        get_packet_size('pkt_size_uni_server', 556, 100),
        1, 0,
        count_1rtt=1,
        count_ack=np.random.randint(0, 2),
        count_new_connection_id=np.random.randint(0, 2),
        count_crypto=np.random.randint(0, 2),
        count_handshake_done=1,
        http3_stream_count=1,
        stream_length=settings_size,
        stream_type_count=1
    ))
    server_app_bytes_sent += settings_size
    
    # Server sends unidirectional control streams
    server_uni_count = blueprint.get('server_uni_streams_count', 4)
    for i in range(max(0, server_uni_count - 1)):
        stream_len = np.random.choice([1, 1, 1, 26, 72], p=[0.5, 0.2, 0.1, 0.1, 0.1])
        is_fin = (i >= server_uni_count - 2) or (np.random.random() > 0.7)
        
        base_size = get_packet_size('pkt_size_uni_server', 92, 20)
        size = base_size + stream_len if stream_len > 1 else base_size
        
        output_packet_features.append(create_packet(
            get_delta_time('data', 's2c'),
            size, 1, 0,
            count_1rtt=1,
            http3_stream_count=1,
            http3_fin_count=1 if is_fin else 0,
            stream_length=stream_len,
            stream_type_count=1
        ))
        server_app_bytes_sent += stream_len
    
    # Client ACK
    output_packet_features.append(create_packet(
        get_delta_time('ack'),
        get_packet_size('ack_client', 91, 10),
        0, 0, count_1rtt=1, count_ack=1
    ))
    
    # =================================================================
    # PHASE 3: PRE-MIGRATION PROBING (if applicable)
    # =================================================================
    if blueprint.get('migration_type', 'NONE') != 'NONE':
        # Optional pre-migration path validation
        if np.random.random() > 0.5:  # 50% chance of early probing
            output_packet_features.append(create_packet(
                get_delta_time('data', 'c2s'),
                get_packet_size('path_challenge', 1398, 50),
                0, 0, count_1rtt=1, count_padding=1, count_path_challenge=1
            ))
            
            output_packet_features.append(create_packet(
                get_delta_time('data', 's2c'),
                get_packet_size('path_response', 1441, 50),
                1, 0, count_1rtt=1
            ))
            
            output_packet_features.append(create_packet(
                get_delta_time('ack'),
                get_packet_size('ack_client', 100, 15),
                0, 0, count_1rtt=1, count_ack=1, count_path_response=1
            ))
            
            output_packet_features.append(create_packet(
                get_delta_time('ack'),
                get_packet_size('ack_server', 91, 10),
                1, 0, count_1rtt=1, count_ack=1
            ))
    
    # =================================================================
    # PHASE 4: CLIENT APPLICATION DATA
    # =================================================================
    # Client sends unidirectional streams (QPACK, etc.)
    client_uni_count = blueprint.get('client_uni_streams_count', 4)
    for i in range(client_uni_count):
        # Variable stream sizes
        if i == 0:
            stream_len = np.random.randint(15, 25)  # SETTINGS-like
        elif i >= client_uni_count - 2:
            stream_len = np.random.choice([26, 72], p=[0.6, 0.4])  # Larger final streams
        else:
            stream_len = np.random.randint(1, 5)  # Small control streams
        
        is_fin = (i >= client_uni_count - 2) or (np.random.random() > 0.6)
        base_size = get_packet_size('pkt_size_uni_client', 110, 30)
        size = base_size + (stream_len if stream_len > 10 else 0)
        
        output_packet_features.append(create_packet(
            get_delta_time('data', 'c2s'),
            size, 0, 0,
            count_1rtt=1,
            http3_stream_count=1,
            http3_fin_count=1 if is_fin else 0,
            stream_length=stream_len,
            stream_type_count=1
        ))
        client_app_bytes_sent += stream_len
    
    # Client sends bidirectional requests
    client_bidi_count = blueprint.get('client_bidi_streams_count', 1)
    avg_request_size = blueprint.get('avg_request_size', 100)
    
    for i in range(client_bidi_count):
        # Variable request sizes around the average
        request_bytes = int(np.random.normal(avg_request_size, avg_request_size * 0.3))
        request_bytes = max(50, request_bytes)
        
        # Maybe piggyback ACK
        has_ack = np.random.random() > 0.5
        
        output_packet_features.append(create_packet(
            get_delta_time('data', 'c2s' if i == 0 else 's2c'),
            get_packet_size('pkt_size_bidi_client', 211, 50) + (request_bytes // 10),
            1 if i > 0 else 0, 0,  # Alternate direction sometimes
            count_1rtt=1,
            count_ack=1 if has_ack else 0,
            http3_stream_count=1,
            http3_fin_count=1,
            stream_length=request_bytes,
            stream_type_count=1
        ))
        client_app_bytes_sent += request_bytes
    
    # =================================================================
    # PHASE 5: CONNECTION MIGRATION
    # =================================================================
    migration_type = blueprint.get('migration_type', 'NONE')
    if migration_type != 'NONE' and not has_migrated:
        has_migrated = True
        
        # Wait until migration time
        time_until_migration = blueprint.get('time_to_migration_msec', 50.0) - current_time_msec
        if time_until_migration > 5.0:
            wait_delta = max(0.001, (time_until_migration / 1000.0) * np.random.uniform(0.8, 1.2))
        else:
            wait_delta = get_delta_time('data', 'c2s')
        
        # 1. Client PATH_CHALLENGE on new path (RFC 9000: padded to >= 1200 bytes)
        output_packet_features.append(create_packet(
            wait_delta,
            get_packet_size('path_challenge', PATH_VALIDATION_MTU_MIN + 100, 100),
            0, 0, count_1rtt=1, count_path_challenge=1, count_padding=1
        ))
        
        # 2. Server PATH_RESPONSE + PATH_CHALLENGE (bidirectional validation)
        validation_rtt = blueprint.get('migration_validation_duration_msec', 20.0) / 1000.0
        half_rtt = (validation_rtt / 2.0) * np.random.uniform(0.8, 1.2)
        
        output_packet_features.append(create_packet(
            half_rtt,
            get_packet_size('path_response', PATH_VALIDATION_MTU_MIN + 150, 100),
            1, 0,
            count_1rtt=1,
            count_path_response=1,
            count_path_challenge=1,
            count_padding=1,
            count_ack=np.random.randint(0, 2)
        ))
        
        # 3. Client PATH_RESPONSE
        output_packet_features.append(create_packet(
            half_rtt,
            get_packet_size('path_response', PATH_VALIDATION_MTU_MIN + 100, 100),
            0, 0,
            count_1rtt=1,
            count_ack=1,
            count_path_response=1,
            count_padding=1
        ))
        
        # 4. Server ACK
        output_packet_features.append(create_packet(
            get_delta_time('ack'),
            get_packet_size('ack_server', 91, 10),
            1, 0, count_1rtt=1, count_ack=1
        ))
    
    # =================================================================
    # PHASE 6: POST-MIGRATION DATA
    # =================================================================
    total_server_bytes = blueprint.get('total_server_app_bytes', 0)
    remaining_server_bytes = total_server_bytes - server_app_bytes_sent
    
    if remaining_server_bytes > 20:
        # Send remaining data in variable-sized packets
        avg_response_size = blueprint.get('avg_response_size', 71)
        num_response_packets = max(1, int(remaining_server_bytes / avg_response_size))
        
        for i in range(num_response_packets):
            bytes_this_packet = min(
                int(np.random.normal(avg_response_size, avg_response_size * 0.4)),
                remaining_server_bytes
            )
            bytes_this_packet = max(10, bytes_this_packet)
            
            is_fin = (i == num_response_packets - 1) or (remaining_server_bytes <= bytes_this_packet)
            
            output_packet_features.append(create_packet(
                get_delta_time('data', 's2c'),
                get_packet_size('pkt_size_bidi_server', 100, 40) + bytes_this_packet,
                1, 0,
                count_1rtt=1,
                http3_stream_count=1 if is_fin else 0,
                http3_fin_count=1 if is_fin else 0,
                stream_length=bytes_this_packet,
                stream_type_count=1 if is_fin else 0
            ))
            
            server_app_bytes_sent += bytes_this_packet
            remaining_server_bytes -= bytes_this_packet
            
            if remaining_server_bytes <= 0:
                break
    
    # =================================================================
    # PHASE 7: CONNECTION CLOSE
    # =================================================================
    close_type = blueprint.get('connection_close_type', 'CLIENT_CLOSE')
    
    # Wait until connection duration
    time_until_close = blueprint.get('connection_duration_msec', 100.0) - current_time_msec
    if time_until_close > 10.0:
        close_delta = max(0.010, (time_until_close / 1000.0) * np.random.uniform(0.9, 1.1))
    else:
        close_delta = get_delta_time('data', 'c2s')
    
    if close_type == 'CLIENT_CLOSE':
        output_packet_features.append(create_packet(
            close_delta,
            get_packet_size('close_client', 97, 15),
            0, 0, count_1rtt=1, count_connection_close=1
        ))
        
        # Server may ACK (not always captured)
        if np.random.random() > 0.3:
            output_packet_features.append(create_packet(
                get_delta_time('ack'),
                get_packet_size('ack_server', 91, 10),
                1, 0, count_1rtt=1, count_ack=1
            ))
    
    elif close_type == 'SERVER_CLOSE':
        output_packet_features.append(create_packet(
            close_delta,
            get_packet_size('close_server', 97, 15),
            1, 0, count_1rtt=1, count_connection_close=1
        ))
        
        if np.random.random() > 0.3:
            output_packet_features.append(create_packet(
                get_delta_time('ack'),
                get_packet_size('ack_client', 91, 10),
                0, 0, count_1rtt=1, count_ack=1
            ))
    
    return output_packet_features


# Example usage
if __name__ == "__main__":
    real_world_blueprint = {
        "connection_duration_msec": 107.25, 
        "retry_occurred": 1, 
        "server_issued_cid_count": 1, 
        "migration_type": "IP_AND_PORT", 
        "connection_close_type": "CLIENT_CLOSE",
        "handshake_duration_msec": 77.66, 
        "total_client_app_bytes": 119, 
        "total_server_app_bytes": 355, 
        "avg_request_size": 23.8, 
        "avg_response_size": 71.0,
        "client_bidi_streams_count": 1, 
        "client_uni_streams_count": 4, 
        "server_uni_streams_count": 4, 
        "time_to_migration_msec": 86.83, 
        "app_data_bytes_before_migration": 47, 
        "migration_validation_duration_msec": 1.17
    }
    
    example_stats = {
        'packet_sizes': {
            'handshake_initial_client': {'mean': 1248, 'std': 10, 'samples': 100},
            'handshake_other_server': {'mean': 517, 'std': 100, 'samples': 50},
            'handshake_other_client': {'mean': 1398, 'std': 50, 'samples': 50},
            'pkt_size_uni_server': {'mean': 100, 'std': 30, 'samples': 200},
            'pkt_size_uni_client': {'mean': 110, 'std': 25, 'samples': 200},
            'pkt_size_bidi_client': {'mean': 211, 'std': 40, 'samples': 100},
            'pkt_size_bidi_server': {'mean': 150, 'std': 50, 'samples': 200},
            'ack_client': {'mean': 91, 'std': 8, 'samples': 500},
            'ack_server': {'mean': 91, 'std': 8, 'samples': 500},
            'path_challenge': {'mean': 1398, 'std': 30, 'samples': 50},
            'path_response': {'mean': 1398, 'std': 30, 'samples': 50},
            'close_client': {'mean': 97, 'std': 10, 'samples': 50},
        },
        'delta_times': {
            'handshake_s2c': {'mean': 0.0010, 'std': 0.0008, 'samples': 100},
            'handshake_c2s': {'mean': 0.0020, 'std': 0.0015, 'samples': 100},
            'server_response': {'mean': 0.0003, 'std': 0.0002, 'samples': 1000},
            'client_request': {'mean': 0.0010, 'std': 0.0008, 'samples': 100},
            'ack_response': {'mean': 0.0003, 'std': 0.0002, 'samples': 500}
        }
    }
    
    # Generate 3 different captures from the same blueprint
    print("Generating 3 varied captures from the same blueprint...\n")
    
    for run in range(3):
        packets = generate_statistically_realistic_features(real_world_blueprint, example_stats)
        
        print(f"=== RUN {run + 1} ===")
        print(f"Generated {len(packets)} packets")
        print(f"Sample packets:")
        for pkt in packets[:3]:
            print(f"  #{pkt['frame_number']}: Δ{pkt['delta_time']:.6f}s, {pkt['packet_length']}B, dir={pkt['packet_direction']}")
        print(f"  ... (middle packets)")
        for pkt in packets[-2:]:
            print(f"  #{pkt['frame_number']}: Δ{pkt['delta_time']:.6f}s, {pkt['packet_length']}B, dir={pkt['packet_direction']}")
        print()
        
        # Save to CSV
        df = pd.DataFrame(packets)
        filename = f'generated_capture_run{run + 1}.csv'
        df.to_csv(filename, index=False)
        print(f"Saved to {filename}\n")

Generating 3 varied captures from the same blueprint...

=== RUN 1 ===
Generated 27 packets
Sample packets:
  #1: Δ0.000000s, 1251B, dir=0
  #2: Δ0.000414s, 90B, dir=1
  #3: Δ0.000265s, 1251B, dir=0
  ... (middle packets)
  #26: Δ0.000391s, 195B, dir=1
  #27: Δ0.105125s, 100B, dir=0

Saved to generated_capture_run1.csv

=== RUN 2 ===
Generated 31 packets
Sample packets:
  #1: Δ0.000000s, 1253B, dir=0
  #2: Δ0.001018s, 91B, dir=1
  #3: Δ0.002434s, 1261B, dir=0
  ... (middle packets)
  #30: Δ0.102868s, 102B, dir=0
  #31: Δ0.000312s, 95B, dir=1

Saved to generated_capture_run2.csv

=== RUN 3 ===
Generated 31 packets
Sample packets:
  #1: Δ0.000000s, 1250B, dir=0
  #2: Δ0.000835s, 72B, dir=1
  #3: Δ0.000001s, 1245B, dir=0
  ... (middle packets)
  #30: Δ0.000001s, 168B, dir=1
  #31: Δ0.111530s, 108B, dir=0

Saved to generated_capture_run3.csv



In [83]:
real_world_blueprint = {
        "connection_duration_msec": 42.25, "retry_occurred": 1, "server_issued_cid_count": 1, "migration_type": "IP_AND_PORT", "connection_close_type": "CLIENT_CLOSE",
        "handshake_duration_msec": 12.66, "total_client_app_bytes": 119, "total_server_app_bytes": 355, "avg_request_size": 23.8, "avg_response_size": 71.0,
        "client_bidi_streams_count": 1, "client_uni_streams_count": 4, "server_uni_streams_count": 4, "time_to_migration_msec": 86.83, "app_data_bytes_before_migration": 47, "migration_validation_duration_msec": 1.17
    }

print("--- Generating features with STATISTICALLY REALISTIC script (Byte Totals Guaranteed) ---")
low_level_packets = generate_statistically_realistic_features(real_world_blueprint, full_stats_profile)
df = pd.DataFrame(low_level_packets)
def create_summary(row):
    info = ['C->S' if row['packet_direction'] == 0 else 'S->C']
    for col, name in [('quic_packet_type_is_initial', 'Initial'), ('quic_packet_type_is_handshake', 'Handshake'),
                        ('quic_packet_type_is_1rtt', '1-RTT'), ('quic_packet_type_is_retry', 'Retry')]:
        if row[col] == 1: info.append(name); break
    for col, name in [('has_new_connection_id', 'NEW_CID'), ('has_path_challenge', 'PATH_CHALLENGE'),
                        ('has_path_response', 'PATH_RESPONSE'), ('has_ack', 'ACK'),
                        ('has_connection_close', 'CLOSE'), ('has_ping', 'PING')]:
        if row[col] > 0: info.append(name)
    if row['has_padding'] == 1: info.append('PADDED')
    if row['http3_stream_count'] > 0: info.append(f"STREAM({row['http3_stream_count']})")
    return ', '.join(info)

display(df)

--- Generating features with STATISTICALLY REALISTIC script (Byte Totals Guaranteed) ---


,frame_number,delta_time,packet_length,packet_direction,header_form,count_initial,count_0rtt,count_handshake,count_1rtt,count_retry,count_vn,count_ack,count_padding,count_connection_close,count_path_challenge,count_path_response,count_new_connection_id,count_retire_cid,count_crypto,count_handshake_done,http3_stream_count,http3_fin_count,stream_length,stream_type_count
0,1,0.000000,1270,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,0.000001,74,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,3,0.001679,1376,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,4,0.007047,79,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,5,0.001176,1215,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
5,6,0.000001,1200,1,1,1,0,0,0,0,0,1,0,0,0,0,0,0,2,0,0,0,0,0
6,7,0.002851,79,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
7,8,0.000001,1203,0,1,0,0,1,0,0,0,1,1,0,0,0,1,0,1,0,0,0,0,0
8,9,0.000001,213,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,1,1,0,17,1
9,10,0.001165,40,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1
